# Figure 3 version 2 — external biological benchmarking

This notebook has exactly two figure-production parts. Each part renders its individual source plots first and compiles the publication figure only afterward.

## Execution contract

Plot-only workflow using frozen scKNIFE/scCellFie results and staged version-2 inputs. Patient-level statistics are recomputed deterministically; upstream models are not.

In [ ]:
PLOT_ONLY_FIGURE = "Figure_3_version_2"
UPSTREAM_MODELS_RECOMPUTED = False
DOWNSTREAM_PATIENT_STATISTICS_RECOMPUTED = True
ALLOWED_INPUT_SUFFIXES = {".h5ad", ".h5", ".csv", ".gz", ".parquet", ".json"}
print(
    PLOT_ONLY_FIGURE,
    "upstream_models_recomputed=", UPSTREAM_MODELS_RECOMPUTED,
    "downstream_patient_statistics_recomputed=", DOWNSTREAM_PATIENT_STATISTICS_RECOMPUTED,
)


## Interpretation boundary

This is external benchmarking, not discovery. Patients are the biological replicates. Pooled-cell GO tests are descriptive localization analyses and are not treated as independent clinical evidence.

# Part I — Main Figure

Prepare the locked benchmarking inputs, show every main-figure source plot separately, and then compile the unchanged Main Figure 3.

## Shared setup and validated inputs

In [ ]:
%matplotlib inline
from pathlib import Path
from typing import Mapping
from dataclasses import replace
from itertools import combinations
import itertools
from IPython.display import display
import base64, hashlib, json, platform, sys, textwrap
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.cm import ScalarMappable
from matplotlib.colors import Normalize
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from PIL import Image
import numpy as np
import pandas as pd
from scipy.stats import mannwhitneyu

ROOT = Path.cwd().resolve()
if not (ROOT / "data/Figure_3_version_2").is_dir():
    raise FileNotFoundError("Run this notebook from source_snapshot/Figures")
DATA_DIR = ROOT / "data/Figure_3_version_2"
OUTPUT_DIR = ROOT / "figs/panel/Figure_3_version_2"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Figure parameters (user editable)
MM = 1 / 25.4
OUTPUT_DPI = 300
RANDOM_SEED = 42
BOOTSTRAP_REPLICATES = 5000
MIN_CELLS_PER_PATIENT_CELLTYPE = 20
MIN_GROUP_PATIENTS_RNR = 4
MIN_GROUP_PATIENTS_ENE = 2
MIN_CONCORDANCE = 0.70
TOP_DOTPLOT_PER_COMPARTMENT = 4
COMPARTMENTS = ("Myeloid", "Tumor/epithelial", "T cell")
FAV_COLOR, UNFAV_COLOR = "#0072B2", "#D55E00"
COHORT_COLORS = {"Bassez A": "#009E73", "Bassez B": "#CC79A7"}
COHORT_MARKERS = {"Bassez A": "o", "Bassez B": "s"}
PANEL_LABEL_SIZE_PT, TITLE_SIZE_PT = 10, 8
AXIS_TITLE_SIZE_PT, TICK_SIZE_PT, NOTE_SIZE_PT = 7, 5.2, 4.6
MAIN_WIDTH_MM, MAIN_HEIGHT_MM = 180, 180
AUDIT_HEIGHT_MM = 245
A4_PORTRAIT_WIDTH_MM, A4_PORTRAIT_HEIGHT_MM = 210, 297
A4_MARGIN_MM, A4_GUTTER_MM = 8, 5
A4_LAYOUT_ROW_GAP_MM = 5
UPSET_BAR_WIDTH = 0.62
UPSET_MATRIX_POINT_SIZE = 13
UPSET_MATRIX_LINE_WIDTH = 0.85
UPSET_INACTIVE_COLOR = "#D9D9D9"
UPSET_ACTIVE_COLOR = "#303030"
UPSET_LEGEND_SIZE_PT = 4.0
UPSET_CELLTYPE_COLORS = {
    "B cell":"#bcbd22", "B_cell":"#bcbd22", "Bcell":"#bcbd22",
    "Cancer cell":"#1f77b4", "Cancer_cell":"#1f77b4", "Epithelial":"#1f77b4",
    "Endothelial cell":"#aec7e8", "Endothelial_cell":"#aec7e8", "Endothelial":"#aec7e8",
    "Fibroblast":"#8c564b", "Fibroblasts":"#8c564b",
    "Myeloid cell":"#e377c2", "Myeloid_cell":"#e377c2", "myeloid":"#e377c2",
    "T cell":"#7f7f7f", "T_cell":"#7f7f7f", "Tcell":"#7f7f7f",
    "Mast cell":"#ff7f0e", "mastcell":"#ff7f0e",
    "PVL":"#9467bd",
}
ACTIVITY_AXIS_LABEL = "Patient median normalized Method 5 activity (%)"
ACTIVITY_AXIS_TICKS = 4
TREATMENT_HEIGHT_MM = DELTA_HEIGHT_MM = 72
MIN_CELLS_PER_PAIRED_TIMEPOINT = 20
PRE_BOX_ALPHA, ON_BOX_ALPHA = 0.20, 0.42
BOX_WIDTH, PATIENT_POINT_SIZE, PATIENT_POINT_EDGE_WIDTH = 0.56, 10, 0.60
PAIR_GAP, BRACKET_LINE_WIDTH = 0.72, 0.65
BRACKET_HEIGHT_FRACTION, BRACKET_OFFSET_FRACTION = 0.025, 0.055
DELTA_POINT_SIZE, DELTA_POINT_EDGE_WIDTH, DELTA_JITTER_WIDTH = 15, 0.70, 0.09
DELTA_MEDIAN_HALF_WIDTH, DELTA_MEDIAN_LINE_WIDTH = 0.20, 1.6
DELTA_AXIS_ROUNDING, DELTA_ZERO_LINE_WIDTH = 0.5, 0.65
DELTA_ZERO_LINE_COLOR = "#777777"
UMAP_BASELINE_HEIGHT_MM, UMAP_LONGITUDINAL_HEIGHT_MM = 118, 224
UMAP_BACKGROUND_SIZE, UMAP_OVERLAY_SIZE = 0.075, 0.24
UMAP_BACKGROUND_ALPHA, UMAP_OVERLAY_ALPHA = 0.30, 0.82
UMAP_BACKGROUND_COLOR, UMAP_CMAP = "#D9D9D9", "Spectral_r"
UMAP_LOW_QUANTILE, UMAP_HIGH_QUANTILE = 0.02, 0.98
UMAP_RASTERIZED = True
UMAP_RASTER_DPI = 300  # raster point layers inside PDF/SVG
FINAL_S3_TITLE_SIZE_PT = 15
# GO heatmap parameters (same selection and clustermap grammar as metabolic tasks)
BASSEZ_GO_HEATMAP_TOP_N = 25
BASSEZ_GO_HEATMAP_PADJ = 0.05
BASSEZ_GO_COMPILED_ROW_FONT_PT = 3.6
FINAL_S3_A_PAGE_PT = (1152, 828)  # 16 x 11.5 inches
S3A_FINAL_SIZE_IN = (16, 11.5)
S3A_GRID_HEIGHT_RATIOS = (0.92, 1.08)
S3A_GRID_HSPACE = 0.16
S3A_PAGE_TITLE_Y = 0.985
S3A_TOP_SECTION_TITLE_Y = 0.925
S3A_BOTTOM_SECTION_TITLE_Y = 0.49
S3A_CANDIDATE_LEFT = 0.17
S3A_SOURCE_BOX_WIDTH_MM, S3A_SOURCE_BOX_HEIGHT_MM = 356, 82
FINAL_S3_B_PAGE_PT = (1152, 1584)  # 16 x 22 inches
S3B_GRID_TOP = 0.91
S3B_GRID_RIGHT = 0.97
S3B_PAGE_TITLE_Y = 0.985
S3B_TOP_PANEL_TITLE_Y = 0.925
S3B_BOTTOM_PANEL_TITLE_Y = 0.465
UMAP_TITLE_SIZE_PT = TITLE_SIZE_PT - 1.1
UMAP_NOTE_SIZE_PT = UMAP_COLORBAR_SIZE_PT = NOTE_SIZE_PT
RAW_SCORE_FLOOR = np.finfo(np.float32).tiny
EPSILON = 1e-10
DISPLAY_TRANSFORMATION = "log10(Method 5 score + 1e-10)"
LITERATURE_PRIORITY = {
    "Myeloid": ["Glycogen degradation", "Glucose to lactate conversion", "Arginine degradation", "Krebs cycle - oxidative decarboxylation of pyruvate"],
    "Tumor/epithelial": ["Arachidonate synthesis", "Palmitate synthesis", "Synthesis of palmitoyl-CoA", "Methionine degradation"],
    "T cell": ["Recognition of misfolded protein", "Heme synthesis", "Methionine degradation", "Oxidative phosphorylation via succinate-coenzyme Q oxidoreductase (COMPLEX II)"],
}

def _style():
    mpl.rcParams.update({"font.family":"sans-serif", "font.sans-serif":["Arial","Helvetica","DejaVu Sans"],
        "axes.grid":False, "axes.spines.top":False, "axes.spines.right":False,
        "pdf.fonttype":42, "ps.fonttype":42, "svg.fonttype":"none", "figure.dpi":OUTPUT_DPI})
_style()

def _sha(path):
    h = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""): h.update(block)
    return h.hexdigest()

def _cliffs_delta(group_b, group_a):
    b = np.asarray(group_b, float); a = np.asarray(group_a, float)
    return float((np.greater.outer(b, a).sum() - np.less.outer(b, a).sum()) / (len(a)*len(b)))

manifest = json.loads((DATA_DIR / "input_manifest.json").read_text())
for relative, record in manifest.items():
    path = DATA_DIR / relative
    assert path.is_file() and _sha(path) == record["sha256"], relative
print(f"Validated {len(manifest)} staged inputs")


## Locked benchmark definitions

In [ ]:
ARACHIDONATE = "Arachidonate synthesis"
ERAD = "Recognition of misfolded protein"
GLYCOGEN = "Glycogen degradation"

TREATMENT_PANELS = [
    {
        "association_id": "BASSEZ_A_MYELOID_GLUTATHIONE",
        "cohort": "Bassez A", "comparison": "Pre→On", "on_timepoint": "On",
        "cell_type": "Myeloid", "task": "Glutathionate synthesis",
        "display_task": "Glutathione synthesis", "group_labels": ("E", "NE"),
        "expected_patients": (5, 7),
    },
    {
        "association_id": "BASSEZ_B_TCELL_GLUTAMINE",
        "cohort": "Bassez B", "comparison": "Pre→On", "on_timepoint": "On",
        "cell_type": "T cell", "task": "Gluconeogenesis from Glutamine",
        "display_task": "Gluconeogenesis from glutamine", "group_labels": ("E", "NE"),
        "expected_patients": (3, 3),
    },
    {
        "association_id": "SHIAO_RTPD1_FIBRO_ALANINE",
        "cohort": "Shiao", "comparison": "Base→RTPD1", "on_timepoint": "RTPD1",
        "cell_type": "Fibroblast", "task": "Synthesis of alanine from glutamine",
        "display_task": "Alanine synthesis from glutamine", "group_labels": ("R", "NR"),
        "expected_patients": (8, 4),
    },
    {
        "association_id": "SHIAO_PD1_TUMOR_BILIRUBIN",
        "cohort": "Shiao", "comparison": "Base→PD1", "on_timepoint": "PD1",
        "cell_type": "Tumor/epithelial", "task": "Synthesis of bilirubin",
        "display_task": "Bilirubin synthesis", "group_labels": ("R", "NR"),
        "expected_patients": (10, 6),
    },
]

EVIDENCE = [
    {
        "association_id": "RNR_TUMOR_ARACHIDONATE", "panel": "S3_A", "endpoint": "R/NR",
        "cohort_scope": "Shiao", "cell_type": "Tumor/epithelial", "task": ARACHIDONATE,
        "direction": "higher in NR", "statistical_rank": 30,
        "evidence_tier": "Direct TNBC mechanism",
        "primary_citation": "Yu et al., Immunity (2025)",
        "citation_url": "https://pmc.ncbi.nlm.nih.gov/articles/PMC11981829/",
        "rationale": "Therapy-resistant TNBC cells synthesize and export arachidonic-acid-rich lipids that promote immune suppression.",
        "limitation": "The inferred task does not measure arachidonate flux, export, or a specific eicosanoid branch.",
        "cross_endpoint_convergence": False,
    },
    {
        "association_id": "RNR_TCELL_ERAD", "panel": "E", "endpoint": "R/NR",
        "cohort_scope": "Shiao", "cell_type": "T cell", "task": ERAD,
        "direction": "higher in NR", "statistical_rank": 27,
        "evidence_tier": "Close immune mechanism",
        "primary_citation": "Ma et al., Cell Metabolism (2019)",
        "citation_url": "https://pubmed.ncbi.nlm.nih.gov/31031094/",
        "rationale": "ER-stress/XBP1 signaling is a demonstrated driver of dysfunctional, exhausted CD8 T cells in tumors.",
        "limitation": "The ERAD task is narrower than the unfolded-protein response and may reflect adaptive proteostasis.",
        "cross_endpoint_convergence": True,
    },
    {
        "association_id": "ENE_MYELOID_GLYCOGEN", "panel": "F", "endpoint": "E/NE",
        "cohort_scope": "Bassez A + B", "cell_type": "Myeloid", "task": GLYCOGEN,
        "direction": "higher in E", "statistical_rank": 3,
        "evidence_tier": "Close macrophage mechanism",
        "primary_citation": "Pahwa et al. (2024); Ma et al., Nature Communications (2020)",
        "citation_url": "https://pmc.ncbi.nlm.nih.gov/articles/PMC11624281/; https://www.nature.com/articles/s41467-020-15636-8",
        "rationale": "Macrophage glycogenolysis supports inflammatory cytokine secretion, phagocytosis, and macrophage activation.",
        "limitation": "This evidence is not specific to TNBC or to clonotype expansion and does not establish clinical response.",
        "cross_endpoint_convergence": False,
    },
    {
        "association_id": "ENE_TCELL_ERAD", "panel": "S3_A", "endpoint": "E/NE",
        "cohort_scope": "Bassez A + B", "cell_type": "T cell", "task": ERAD,
        "direction": "higher in NE", "statistical_rank": 2,
        "evidence_tier": "Close immune mechanism",
        "primary_citation": "Ma et al., Cell Metabolism (2019)",
        "citation_url": "https://pubmed.ncbi.nlm.nih.gov/31031094/",
        "rationale": "ER-stress/XBP1 signaling is a demonstrated driver of dysfunctional, exhausted CD8 T cells in tumors.",
        "limitation": "E/NE is a clonotype-expansion endpoint, not clinical response; the ERAD score is not a direct exhaustion assay.",
        "cross_endpoint_convergence": True,
    },
]

## Patient-level baseline screen

In [ ]:
def _bh_adjust(values):
    values = np.asarray(values, float)
    finite = np.isfinite(values)
    out = np.full(values.shape, np.nan)
    if not finite.any():
        return out
    p = values[finite]
    order = np.argsort(p)
    ranked = p[order]
    adjusted = np.minimum.accumulate(
        (ranked * len(ranked) / np.arange(1, len(ranked) + 1))[::-1]
    )[::-1]
    restored = np.empty_like(adjusted)
    restored[order] = np.clip(adjusted, 0, 1)
    out[finite] = restored
    return out


def _exact_permutation_p(group_a, group_b):
    group_a = np.asarray(group_a, float)
    group_b = np.asarray(group_b, float)
    values = np.concatenate([group_a, group_b])
    observed = abs(np.median(group_b) - np.median(group_a))
    indices = np.arange(len(values))
    extreme = total = 0
    for a_index in itertools.combinations(indices, len(group_a)):
        mask = np.zeros(len(values), bool)
        mask[np.asarray(a_index, int)] = True
        statistic = abs(np.median(values[~mask]) - np.median(values[mask]))
        extreme += statistic >= observed - 1e-12
        total += 1
    return extreme / total if total else np.nan


def _bootstrap_ci(group_a, group_b, seed):
    group_a = np.asarray(group_a, float)
    group_b = np.asarray(group_b, float)
    rng = np.random.default_rng(seed)
    values = np.empty(BOOTSTRAP_REPLICATES)
    for index in range(BOOTSTRAP_REPLICATES):
        a = rng.choice(group_a, len(group_a), replace=True)
        b = rng.choice(group_b, len(group_b), replace=True)
        values[index] = _cliffs_delta(b, a)
    return tuple(np.quantile(values, [0.025, 0.975]))


def _patient_concordance(group_a, group_b):
    """Fraction lying beyond the opposite-group median in the observed direction."""
    group_a = np.asarray(group_a, float)
    group_b = np.asarray(group_b, float)
    direction = np.sign(np.median(group_b) - np.median(group_a))
    if direction == 0:
        return 0, len(group_a) + len(group_b), 0.0
    if direction > 0:
        flags = np.r_[group_a < np.median(group_b), group_b > np.median(group_a)]
    else:
        flags = np.r_[group_a > np.median(group_b), group_b < np.median(group_a)]
    return int(flags.sum()), len(flags), float(flags.mean())


def _screen_one_cohort(patient, cohort, group_a_label, group_b_label, min_group):
    rows = []
    block = patient.loc[patient.cohort.eq(cohort)]
    for (cell_type, task), task_block in block.groupby(["cell_type", "task"], observed=True, sort=False):
        a = task_block.loc[task_block.status.eq(group_a_label), "log2_median_score"].to_numpy()
        b = task_block.loc[task_block.status.eq(group_b_label), "log2_median_score"].to_numpy()
        eligible = len(a) >= min_group and len(b) >= min_group and np.unique(np.r_[a, b]).size > 1
        if eligible:
            delta = _cliffs_delta(b, a)
            p_value = _exact_permutation_p(a, b)
            low, high = _bootstrap_ci(a, b, RANDOM_SEED + len(rows))
            n_concordant, n_total, concordance = _patient_concordance(a, b)
        else:
            delta = p_value = low = high = concordance = np.nan
            n_concordant, n_total = 0, len(a) + len(b)
        rows.append({
            "endpoint": "R/NR" if cohort == "Shiao" else "E/NE",
            "cohort": cohort, "cell_type": cell_type, "task": task,
            "group_a": group_a_label, "group_b": group_b_label,
            "n_group_a": len(a), "n_group_b": len(b), "eligible": eligible,
            "delta_group_b_vs_a": delta, "ci_low": low, "ci_high": high,
            "exact_permutation_p": p_value, "n_patient_concordant": n_concordant,
            "n_patients": n_total, "patient_concordance_rate": concordance,
        })
    result = pd.DataFrame(rows)
    result["exact_permutation_q"] = _bh_adjust(result.exact_permutation_p)
    return result


def _literature_rank(cell_type, task):
    priorities = LITERATURE_PRIORITY[cell_type]
    return priorities.index(task) if task in priorities else len(priorities) + 1


def _select_rnr(screen):
    selected = []
    ranked_rows = []
    for cell_type in COMPARTMENTS:
        block = screen.loc[screen.cell_type.eq(cell_type) & screen.eligible].copy()
        block["abs_delta"] = block.delta_group_b_vs_a.abs()
        block["concordance_pass"] = block.patient_concordance_rate.ge(MIN_CONCORDANCE)
        candidates = block.loc[block.concordance_pass].copy()
        if candidates.empty:
            candidates = block.copy()
        max_effect = candidates.abs_delta.max()
        candidates["effect_top_tier"] = candidates.abs_delta.ge(max_effect - .10)
        candidates["literature_rank"] = [
            _literature_rank(cell_type, task) for task in candidates.task
        ]
        candidates = candidates.sort_values(
            ["effect_top_tier", "literature_rank", "abs_delta", "patient_concordance_rate", "task"],
            ascending=[False, True, False, False, True],
        )
        chosen = candidates.iloc[0].copy()
        selected.append(chosen)
        block = block.sort_values(["abs_delta", "patient_concordance_rate", "task"], ascending=[False, False, True])
        block["endpoint_rank"] = np.arange(1, len(block) + 1)
        block["selected"] = block.task.eq(chosen.task)
        ranked_rows.append(block)
    return pd.DataFrame(selected), pd.concat(ranked_rows, ignore_index=True)


def _select_ene(screen):
    wide = screen.pivot(index=["cell_type", "task"], columns="cohort", values=[
        "eligible", "delta_group_b_vs_a", "ci_low", "ci_high", "exact_permutation_p",
        "exact_permutation_q", "patient_concordance_rate", "n_group_a", "n_group_b",
    ])
    wide.columns = [f"{metric}__{cohort}" for metric, cohort in wide.columns]
    wide = wide.reset_index()
    wide["same_direction"] = np.sign(wide["delta_group_b_vs_a__Bassez A"]).eq(
        np.sign(wide["delta_group_b_vs_a__Bassez B"])
    ) & wide["delta_group_b_vs_a__Bassez A"].ne(0)
    wide["eligible"] = (
        wide["eligible__Bassez A"].astype(bool)
        & wide["eligible__Bassez B"].astype(bool)
        & wide.same_direction
    )
    wide["min_abs_delta"] = wide[[
        "delta_group_b_vs_a__Bassez A", "delta_group_b_vs_a__Bassez B"
    ]].abs().min(axis=1)
    wide["mean_concordance"] = wide[[
        "patient_concordance_rate__Bassez A", "patient_concordance_rate__Bassez B"
    ]].mean(axis=1)
    selected = []
    ranked_rows = []
    for cell_type in COMPARTMENTS:
        block = wide.loc[wide.cell_type.eq(cell_type) & wide.eligible].copy()
        block["concordance_pass"] = block.mean_concordance.ge(MIN_CONCORDANCE)
        candidates = block.loc[block.concordance_pass].copy()
        if candidates.empty:
            candidates = block.copy()
        max_effect = candidates.min_abs_delta.max()
        candidates["effect_top_tier"] = candidates.min_abs_delta.ge(max_effect - .10)
        candidates["literature_rank"] = [
            _literature_rank(cell_type, task) for task in candidates.task
        ]
        candidates = candidates.sort_values(
            ["effect_top_tier", "literature_rank", "min_abs_delta", "mean_concordance", "task"],
            ascending=[False, True, False, False, True],
        )
        chosen = candidates.iloc[0].copy()
        selected.append(chosen)
        block = block.sort_values(["min_abs_delta", "mean_concordance", "task"], ascending=[False, False, True])
        block["endpoint_rank"] = np.arange(1, len(block) + 1)
        block["selected"] = block.task.eq(chosen.task)
        ranked_rows.append(block)
    return pd.DataFrame(selected), pd.concat(ranked_rows, ignore_index=True)

In [ ]:
patient = pd.read_csv(DATA_DIR / "eligible_patient_scores.csv.gz")
assert patient.n_cells.ge(MIN_CELLS_PER_PATIENT_CELLTYPE).all()
rnr_screen = _screen_one_cohort(patient, "Shiao", "Favorable", "Unfavorable", MIN_GROUP_PATIENTS_RNR)
ene_screen = pd.concat([
    _screen_one_cohort(patient, cohort, "Favorable", "Unfavorable", MIN_GROUP_PATIENTS_ENE)
    for cohort in ("Bassez A", "Bassez B")
], ignore_index=True)
selected_rnr, ranked_rnr = _select_rnr(rnr_screen)
selected_ene, ranked_ene = _select_ene(ene_screen)

analysis_tables = {
    "rnr_full_screen.csv": rnr_screen,
    "ene_full_screen.csv": ene_screen,
    "rnr_ranked_candidates.csv": ranked_rnr,
    "ene_ranked_candidates.csv": ranked_ene,
    "rnr_selected_tasks.csv": selected_rnr,
    "ene_selected_tasks.csv": selected_ene,
}
for name, frame in analysis_tables.items():
    frame.to_csv(DATA_DIR / name, index=False)

V8_PATHS = {
    "eligible_patient_scores.csv.gz": DATA_DIR / "eligible_patient_scores.csv.gz",
    **{name: DATA_DIR / name for name in analysis_tables},
}
def prepare_v8_data(root, force=False):
    return V8_PATHS

display(rnr_screen.loc[rnr_screen.task.isin(["Arachidonate synthesis", "Recognition of misfolded protein"])])
display(ene_screen.loc[ene_screen.task.isin(["Glycogen degradation", "Recognition of misfolded protein"])])


## Method-overlap analysis from frozen DE tables

In [ ]:
def _read_de(path):
    frame = pd.read_csv(path)
    return frame.loc[:, ~frame.columns.str.match(r"^Unnamed")].copy()

def _overlap_summary(knife, sccellfie, label):
    knife = knife.loc[knife.adj_p_value.lt(0.05)]
    sccellfie = sccellfie.loc[sccellfie.adj_p_value.lt(0.05)]
    rows = []
    for cell_type in sorted(set(knife.cell_type) | set(sccellfie.cell_type)):
        kset = set(knife.loc[knife.cell_type.eq(cell_type), "feature"])
        sset = set(sccellfie.loc[sccellfie.cell_type.eq(cell_type), "feature"])
        rows.append({"comparison":label, "cell_type":cell_type, "Shared":len(kset&sset),
                     "scKNIFE only":len(kset-sset), "scCellFie only":len(sset-kset)})
    return pd.DataFrame(rows)

bknife = _read_de(DATA_DIR / "de/bassez_B_knife.csv")
overlap_parts = [
    _overlap_summary(bknife.loc[bknife.expansion_group.eq("E")], _read_de(DATA_DIR / "de/bassez_B_sccellfie_E.csv"), "Bassez B · E"),
    _overlap_summary(bknife.loc[bknife.expansion_group.eq("NE")], _read_de(DATA_DIR / "de/bassez_B_sccellfie_NE.csv"), "Bassez B · NE"),
    _overlap_summary(_read_de(DATA_DIR / "de/shiao_knife_R.csv"), _read_de(DATA_DIR / "de/shiao_sccellfie_R.csv"), "Shiao · R"),
    _overlap_summary(_read_de(DATA_DIR / "de/shiao_knife_NR.csv"), _read_de(DATA_DIR / "de/shiao_sccellfie_NR.csv"), "Shiao · NR"),
]
PANEL_A_SUMMARY = pd.concat(overlap_parts, ignore_index=True)
# Match the frozen earlier UpSet display scope: Shiao used the seven annotated
# compartments shown in Figure_3_A and did not include the generic myeloid row.
PANEL_A_SUMMARY = PANEL_A_SUMMARY.loc[
    ~(PANEL_A_SUMMARY.comparison.str.startswith("Shiao") & PANEL_A_SUMMARY.cell_type.eq("myeloid"))
].reset_index(drop=True)
celltype_aliases = {
    "B_cell":"B cell", "Bcell":"B cell", "Cancer_cell":"Cancer cell",
    "Endothelial_cell":"Endothelial cell", "Endothelial":"Endothelial cell",
    "Fibroblasts":"Fibroblast", "Myeloid_cell":"Myeloid cell",
    "T_cell":"T cell", "Tcell":"T cell", "mastcell":"Mast cell",
}
PANEL_A_SUMMARY["cell_type"] = PANEL_A_SUMMARY.cell_type.replace(celltype_aliases)
PANEL_A_SUMMARY = PANEL_A_SUMMARY.groupby(
    ["comparison", "cell_type"], sort=False, as_index=False
)[["Shared", "scKNIFE only", "scCellFie only"]].sum()
PANEL_A_SUMMARY.to_csv(DATA_DIR / "panel_a_overlap_summary.csv", index=False)
display(PANEL_A_SUMMARY.groupby("comparison")[["Shared","scKNIFE only","scCellFie only"]].sum())

def _upset(ax, root):
    ax.set_axis_off()
    comparisons = PANEL_A_SUMMARY.comparison.drop_duplicates().tolist()
    panel_titles = {
        "Bassez B · E":"A  Bassez B · E",
        "Bassez B · NE":"B  Bassez B · NE",
        "Shiao · R":"C  Shiao · R",
        "Shiao · NR":"D  Shiao · NR",
    }
    intersections = ["Shared", "scKNIFE only", "scCellFie only"]
    memberships = np.array([[True, True, False], [True, False, True]])
    x = np.arange(len(intersections))

    displayed_celltypes = PANEL_A_SUMMARY.cell_type.drop_duplicates().tolist()
    missing_colors = sorted(set(displayed_celltypes) - set(UPSET_CELLTYPE_COLORS))
    if missing_colors:
        raise ValueError(f"Missing UpSet cell-type colors: {missing_colors}")
    legend_handles = [
        Patch(facecolor=UPSET_CELLTYPE_COLORS[cell_type], label=cell_type)
        for cell_type in displayed_celltypes
    ]
    ax.legend(
        handles=legend_handles, loc="upper center", bbox_to_anchor=(0.5, 1.01),
        ncol=len(legend_handles), frameon=False, fontsize=UPSET_LEGEND_SIZE_PT,
        handlelength=0.9, columnspacing=0.75, borderaxespad=0,
    )

    for index, comparison in enumerate(comparisons):
        container = ax.inset_axes([0.006 + index * 0.249, 0.00, 0.236, 0.80])
        container.set_axis_off()
        block = PANEL_A_SUMMARY.loc[PANEL_A_SUMMARY.comparison.eq(comparison)].copy()
        totals = block[intersections].sum().astype(int)
        union_total = int(totals.sum())
        container.text(
            0, 1.01, panel_titles[comparison], transform=container.transAxes,
            ha="left", va="bottom", fontsize=TITLE_SIZE_PT, fontweight="bold",
        )
        container.text(
            0, 0.91, f"Union = {union_total:,} cell type–task pairs",
            transform=container.transAxes, ha="left", va="bottom",
            fontsize=NOTE_SIZE_PT, color="#444444",
        )

        bar_ax = container.inset_axes([0.24, 0.40, 0.74, 0.46])
        bottom = np.zeros(len(intersections), dtype=float)
        for _, row in block.iterrows():
            values = row[intersections].to_numpy(dtype=float)
            bar_ax.bar(
                x, values, bottom=bottom, width=UPSET_BAR_WIDTH,
                color=UPSET_CELLTYPE_COLORS[row.cell_type], edgecolor="white", linewidth=0.35,
            )
            bottom += values
        for xpos, total in zip(x, totals):
            bar_ax.annotate(
                f"{int(total):,}", (xpos, total), xytext=(0, 1.5), textcoords="offset points",
                ha="center", va="bottom", fontsize=NOTE_SIZE_PT,
            )
        bar_ax.set_xticks([])
        bar_ax.set_ylabel("Intersection size", fontsize=NOTE_SIZE_PT, labelpad=1)
        bar_ax.tick_params(axis="y", labelsize=NOTE_SIZE_PT, width=0.5, length=2, pad=1)
        bar_ax.margins(y=0.16)
        bar_ax.grid(False)
        for spine in ("top", "right"):
            bar_ax.spines[spine].set_visible(False)
        for spine in ("left", "bottom"):
            bar_ax.spines[spine].set_linewidth(0.5)

        matrix_ax = container.inset_axes([0.24, 0.02, 0.74, 0.28])
        for column in range(len(intersections)):
            active = np.flatnonzero(memberships[:, column])
            if len(active) > 1:
                matrix_ax.plot(
                    [column, column], [active.min(), active.max()],
                    color=UPSET_ACTIVE_COLOR, linewidth=UPSET_MATRIX_LINE_WIDTH, zorder=1,
                )
            for row_index in range(2):
                matrix_ax.scatter(
                    column, row_index, s=UPSET_MATRIX_POINT_SIZE,
                    color=UPSET_ACTIVE_COLOR if memberships[row_index, column] else UPSET_INACTIVE_COLOR,
                    edgecolors="none", zorder=2,
                )
        matrix_ax.set_xlim(-0.45, len(intersections) - 0.55)
        matrix_ax.set_ylim(-0.45, 1.45)
        matrix_ax.invert_yaxis()
        matrix_ax.set_xticks(x, ["Shared", "scKNIFE\nonly", "scCellFie\nonly"], fontsize=NOTE_SIZE_PT)
        matrix_ax.set_yticks([0, 1], ["scKNIFE", "scCellFie"], fontsize=NOTE_SIZE_PT)
        matrix_ax.tick_params(length=0, pad=1)
        matrix_ax.set_title("Set membership", loc="left", fontsize=NOTE_SIZE_PT, pad=1)
        for spine in matrix_ax.spines.values():
            spine.set_visible(False)


## Longitudinal patient analysis

In [ ]:
treatment_patient = pd.read_csv(DATA_DIR / "treatment_patient_scores.csv.gz")
treatment_reference = pd.read_csv(DATA_DIR / "treatment_panel_statistics_reference.csv")
recomputed_rows = []
for item in TREATMENT_PANELS:
    block = treatment_patient.loc[treatment_patient.association_id.eq(item["association_id"])]
    wide = block.pivot(index=["patient","status"], columns="timepoint_order", values="log2_median_score").reset_index()
    wide["delta"] = wide[1] - wide[0]
    fav = wide.loc[wide.status.eq("Favorable"), "delta"].to_numpy()
    unfav = wide.loc[wide.status.eq("Unfavorable"), "delta"].to_numpy()
    pvalue = float(mannwhitneyu(fav, unfav, alternative="two-sided", method="exact").pvalue)
    for status, label, values in zip(("Favorable","Unfavorable"), item["group_labels"], (fav,unfav)):
        reference = treatment_reference.loc[
            treatment_reference.association_id.eq(item["association_id"]) & treatment_reference.status.eq(status)
        ].iloc[0].to_dict()
        median = float(np.median(values))
        reference.update(patient_median_delta=median,
                         patient_concordance=float(np.mean(np.sign(values)==np.sign(median))),
                         patient_interaction_p=pvalue, group_label=label)
        recomputed_rows.append(reference)
treatment_statistics = pd.DataFrame(recomputed_rows)
assert np.allclose(treatment_statistics.patient_interaction_p, treatment_reference.patient_interaction_p)
treatment_statistics.to_csv(DATA_DIR / "treatment_panel_statistics.csv", index=False)
display(treatment_statistics[["association_id","group_label","n_patients","patient_median_delta","patient_concordance","patient_interaction_p","de_adj_p_value"]])


## Biological evidence and statistical context

In [ ]:
baseline_effects = pd.read_csv(DATA_DIR / "literature_supported_effects_reference.csv")
treatment_literature = {
    "BASSEZ_A_MYELOID_GLUTATHIONE": {"panel":"G", "evidence_tier":"Close myeloid redox mechanism", "primary_citation":"Roux et al., PNAS (2019)", "citation_url":"https://pubmed.ncbi.nlm.nih.gov/30770442/", "rationale":"Macrophage redox state and ROS regulate PD-L1-linked immune suppression in TNBC.", "limitation":"Glutathione task activity is inferred and does not measure redox flux or PD-L1 causality."},
    "BASSEZ_B_TCELL_GLUTAMINE": {"panel":"H", "evidence_tier":"Close T-cell nutrient mechanism", "primary_citation":"Carr et al., Journal of Immunology (2010)", "citation_url":"https://pubmed.ncbi.nlm.nih.gov/20554958/", "rationale":"Glutamine availability supports activated T-cell growth, proliferation, and function.", "limitation":"The inferred task is glutamine-linked carbon use, not a direct gluconeogenic-flux assay."},
    "SHIAO_RTPD1_FIBRO_ALANINE": {"panel":"I", "evidence_tier":"Plausible stromal nutrient mechanism", "primary_citation":"Sousa et al., Nature (2016); He et al., Clinical and Translational Medicine (2024)", "citation_url":"https://pubmed.ncbi.nlm.nih.gov/27509858/; https://pubmed.ncbi.nlm.nih.gov/39690134/", "rationale":"Stromal alanine and glutamine coupling can support tumor metabolic adaptation.", "limitation":"Evidence is indirect for TNBC RTPD1 response and does not establish alanine transfer or flux."},
    "SHIAO_PD1_TUMOR_BILIRUBIN": {"panel":"J", "evidence_tier":"Direct/close heme-catabolism mechanism", "primary_citation":"Williams et al., Communications Biology (2026); Khojandi et al., Cancer Immunology Research (2021)", "citation_url":"https://pubmed.ncbi.nlm.nih.gov/42106562/; https://pubmed.ncbi.nlm.nih.gov/33303575/", "rationale":"Breast-cancer heme catabolism and bilirubin can suppress macrophage function and reduce anti-PD1 efficacy.", "limitation":"The inferred bilirubin-synthesis task does not directly measure heme turnover or extracellular bilirubin."},
}
table_rows = []
for item in EVIDENCE:
    block = baseline_effects.loc[baseline_effects.association_id.eq(item["association_id"])]
    statistics_text = []
    for row in block.itertuples(index=False):
        statistics_text.append(f"{row.cohort}: n={int(row.n_group_a)}+{int(row.n_group_b)}, delta={row.delta_group_b_vs_a:+.3f}, 95% CI [{row.ci_low:+.3f}, {row.ci_high:+.3f}], p={row.exact_permutation_p:.4g}, q={row.exact_permutation_q:.4g}, concordance={int(row.n_patient_concordant)}/{int(row.n_patients)}")
    table_rows.append({"main_panel":item["panel"], "association_id":item["association_id"], "analysis_type":"Baseline patient comparison", "cohort":item["cohort_scope"], "endpoint":item["endpoint"], "cell_type":item["cell_type"], "task":item["task"], "direction":item["direction"], "evidence_tier":item["evidence_tier"], "primary_citation":item["primary_citation"], "citation_url":item["citation_url"], "rationale":item["rationale"], "limitation":item["limitation"], "patient_level_statistics":" | ".join(statistics_text)})
for item in TREATMENT_PANELS:
    literature = treatment_literature[item["association_id"]]
    block = treatment_statistics.loc[treatment_statistics.association_id.eq(item["association_id"])]
    statistics_text = [f"{row.group_label}: n={int(row.n_patients)}, median delta={row.patient_median_delta:+.3f}, concordance={row.patient_concordance:.3f}, pooled-cell DE q={row.de_adj_p_value:.4g}" for row in block.itertuples(index=False)]
    statistics_text.append(f"exact between-group delta p={block.patient_interaction_p.iloc[0]:.4g}")
    table_rows.append({"main_panel":literature["panel"], "association_id":item["association_id"], "analysis_type":"Paired longitudinal patient change", "cohort":item["cohort"], "endpoint":"E/NE" if item["cohort"].startswith("Bassez") else "R/NR", "cell_type":item["cell_type"], "task":item["display_task"], "direction":"Treatment-minus-baseline change compared between groups", "evidence_tier":literature["evidence_tier"], "primary_citation":literature["primary_citation"], "citation_url":literature["citation_url"], "rationale":literature["rationale"], "limitation":literature["limitation"], "patient_level_statistics":" | ".join(statistics_text)})
evidence_table = pd.DataFrame(table_rows).sort_values("main_panel").reset_index(drop=True)
assert evidence_table.main_panel.tolist() == ["E","F","G","H","I","J","S3_A","S3_A"]
evidence_table_path = DATA_DIR / "Figure_3_version_2_Table_S3_1_biological_evidence.csv"
evidence_table.to_csv(evidence_table_path, index=False)
display(evidence_table)


## Shared publication renderers

In [ ]:
def _cliffs_delta(group_b, group_a):
    b = np.asarray(group_b, float); a = np.asarray(group_a, float)
    return float((np.greater.outer(b, a).sum() - np.less.outer(b, a).sum()) / (len(a)*len(b)))

UMAP_PATHS = {
    "background": DATA_DIR / "task_umap_background.parquet",
    "overlay": DATA_DIR / "task_umap_activity.parquet",
    "metadata": DATA_DIR / "task_umap_metadata.csv",
    "audit": DATA_DIR / "task_umap_validation.csv",
    "render_audit": DATA_DIR / "task_umap_render_audit.csv",
}
PLOT_PATHS = {
    "evidence": DATA_DIR / "literature_supported_tasks.csv",
    "effects": DATA_DIR / "literature_supported_effects_reference.csv",
    "treatment_patient": DATA_DIR / "treatment_patient_scores.csv.gz",
    "treatment_statistics": DATA_DIR / "treatment_panel_statistics.csv",
}


def _short_task(task, width=35):
    return task if len(task) <= width else task[:width - 1] + "…"


def _dotplot(ax, letter, title, ranked, metric, direction_labels):
    blocks = []
    for cell_type in COMPARTMENTS:
        block = ranked.loc[ranked.cell_type.eq(cell_type)].head(TOP_DOTPLOT_PER_COMPARTMENT).copy()
        blocks.append(block)
    data = pd.concat(blocks, ignore_index=True)
    data["label"] = data.cell_type.str.replace("Tumor/epithelial", "Tumor", regex=False) + " · " + data.task.map(_short_task)
    data = data.iloc[::-1].reset_index(drop=True)
    y = np.arange(len(data))
    direction_value = data["delta_group_b_vs_a"] if "delta_group_b_vs_a" in data else data["delta_group_b_vs_a__Bassez A"]
    colors = np.where(direction_value.gt(0), UNFAV_COLOR, FAV_COLOR)
    sizes = 16 + 75 * data[metric].clip(0, 1)
    ax.scatter(data.patient_concordance_rate if "patient_concordance_rate" in data else data.mean_concordance,
               y, s=sizes, c=colors, edgecolor="white", lw=.45)
    chosen = data.selected.to_numpy(bool)
    x_values = (data.patient_concordance_rate if "patient_concordance_rate" in data else data.mean_concordance).to_numpy()
    ax.scatter(x_values[chosen], y[chosen], s=sizes[chosen] + 22, facecolor="none", edgecolor="#111111", lw=.8)
    ax.axvline(MIN_CONCORDANCE, color="#777777", lw=.7, ls="--")
    ax.set_xlim(.45, 1.02)
    ax.set_yticks(y, data.label, fontsize=NOTE_SIZE_PT)
    ax.set_xlabel("Patient directional concordance", fontsize=AXIS_TITLE_SIZE_PT)
    ax.set_title(f"{letter}  {title}", loc="left", fontsize=TITLE_SIZE_PT, fontweight="bold")
    ax.text(.02, .98,
            f"Blue: higher {direction_labels[0]}  ·  orange: higher {direction_labels[1]}  ·  ring: selected",
            transform=ax.transAxes, ha="left", va="top", fontsize=NOTE_SIZE_PT, color="#555555")
    ax.tick_params(axis="x", labelsize=TICK_SIZE_PT)
    for spine in ("top", "right", "left"):
        ax.spines[spine].set_visible(False)
    ax.grid(False)


def _box_panel(ax, letter, cell_type, task, patient, cohorts, group_labels, screen, endpoint_title, compact=False, show_ylabel=True):
    arrays, positions, colors, xticks, xticklabels = [], [], [], [], []
    position = 0
    rng = np.random.default_rng(RANDOM_SEED + ord(letter))
    for cohort in cohorts:
        block = patient.loc[
            patient.cohort.eq(cohort) & patient.cell_type.eq(cell_type) & patient.task.eq(task)
        ]
        for status, label, color in zip(("Favorable", "Unfavorable"), group_labels, (FAV_COLOR, UNFAV_COLOR)):
            values = block.loc[block.status.eq(status), "log2_median_score"].to_numpy()
            arrays.append(values)
            positions.append(position)
            colors.append(color)
            ax.scatter(position + rng.uniform(-.12, .12, len(values)), values, s=10,
                       facecolor="white", edgecolor=color, lw=.6, zorder=3)
            position += 1
        xticks.append(position - 1.5)
        xticklabels.append(cohort.replace("Bassez ", "B"))
        position += .8
    boxes = ax.boxplot(arrays, positions=positions, widths=.58, patch_artist=True, showfliers=False,
                       medianprops={"color": "#222222", "linewidth": .8},
                       whiskerprops={"linewidth": .65}, capprops={"linewidth": .65},
                       boxprops={"linewidth": .65})
    for box, color in zip(boxes["boxes"], colors):
        box.set_facecolor(color)
        box.set_alpha(.38)
    ax.set_xticks(xticks, xticklabels, fontsize=TICK_SIZE_PT)
    ax.set_ylabel(
        "Patient median log2 score" if show_ylabel else "",
        fontsize=AXIS_TITLE_SIZE_PT,
    )
    title_width = 20 if compact else 28
    wrapped_task = "\n".join(textwrap.wrap(task, width=title_width, break_long_words=False))
    ax.set_title(
        f"{letter}  {cell_type} · {endpoint_title}\n{wrapped_task}", loc="left",
        fontsize=TITLE_SIZE_PT - (1.5 if compact else 1), fontweight="bold", pad=3,
    )
    if len(cohorts) == 1:
        row = screen.loc[screen.cell_type.eq(cell_type) & screen.task.eq(task)].iloc[0]
        summary = f"δ={row.delta_group_b_vs_a:+.2f}; p={row.exact_permutation_p:.2g}; q={row.exact_permutation_q:.2g}"
    else:
        parts = []
        for cohort in cohorts:
            row = screen.loc[
                screen.cohort.eq(cohort) & screen.cell_type.eq(cell_type) & screen.task.eq(task)
            ].iloc[0]
            parts.append(f"{cohort.replace('Bassez ', 'B')}: δ={row.delta_group_b_vs_a:+.2f}, q={row.exact_permutation_q:.2g}")
        summary = ("\n" if compact else "  ").join(parts)
    ax.text(
        .5, -.24 if compact else -.20, summary, transform=ax.transAxes,
        ha="center", fontsize=NOTE_SIZE_PT - (0.3 if compact else 0),
    )
    ax.tick_params(axis="y", labelsize=TICK_SIZE_PT)
    for spine in ("top", "right"):
        ax.spines[spine].set_visible(False)
    ax.grid(False)


def _save_v8(fig, stem):
    result = {}
    for extension in ("png", "pdf", "svg", "tiff"):
        path = stem.with_suffix("." + extension)
        fig.savefig(path, dpi=OUTPUT_DPI, bbox_inches="tight", facecolor="white")
        result[extension] = str(path)
    plt.close(fig)
    return result

def _evidence_summary(ax, evidence, effects):
    labels = [
        "Tumor · arachidonate synthesis · R/NR",
        "T cell · ERAD/misfolded protein · R/NR",
        "Myeloid · glycogen degradation · E/NE",
        "T cell · ERAD/misfolded protein · E/NE",
    ]
    y = np.arange(4)[::-1]
    for yy, (_, item) in zip(y, evidence.iterrows()):
        block = effects.loc[effects.association_id.eq(item.association_id)]
        concordance = block.patient_concordance_rate.mean()
        color = UNFAV_COLOR if "higher in N" in item.direction else FAV_COLOR
        ax.scatter(concordance, yy, s=70, color=color, edgecolor="white", lw=.6, zorder=3)
        ax.text(1.01, yy, item.evidence_tier.replace(" mechanism", ""), va="center", ha="right",
                transform=ax.get_yaxis_transform(), fontsize=NOTE_SIZE_PT, color="#555555")
    ax.axvline(.70, color="#888888", ls="--", lw=.7)
    ax.set_xlim(.60, 1.02)
    ax.set_yticks(y, labels, fontsize=NOTE_SIZE_PT + .2)
    ax.set_xlabel("Patient directional concordance", fontsize=7)
    ax.set_title("B  Literature-supported associations", loc="left", fontsize=TITLE_SIZE_PT, fontweight="bold")
    ax.text(.01, 1.01, "Blue: higher favorable  ·  orange: higher unfavorable",
            transform=ax.transAxes, fontsize=NOTE_SIZE_PT, color="#555555", va="bottom")
    ax.text(.01, -.31, "Repeated T-cell ERAD = cross-endpoint convergence (distinct endpoints; not clinical replication).",
            transform=ax.transAxes, fontsize=NOTE_SIZE_PT, color="#555555")
    for spine in ("top", "right", "left"):
        ax.spines[spine].set_visible(False)
    ax.tick_params(axis="x", labelsize=TICK_SIZE_PT)


def _forest(ax, letter, title, effects, endpoint):
    data = effects.loc[effects.endpoint.eq(endpoint)].copy()
    if endpoint == "R/NR":
        data["label"] = data.cell_type.str.replace("Tumor/epithelial", "Tumor", regex=False) + " · " + data.task
        palette = {"Shiao": "#333333"}
        markers = {"Shiao": "o"}
    else:
        data["label"] = data.cell_type + " · " + data.task
        palette, markers = COHORT_COLORS, COHORT_MARKERS
    labels = list(dict.fromkeys(data.label))
    y_base = {label: len(labels) - 1 - i for i, label in enumerate(labels)}
    offsets = {"Shiao": 0, "Bassez A": .11, "Bassez B": -.11}
    for _, row in data.iterrows():
        yy = y_base[row.label] + offsets[row.cohort]
        value = row.delta_group_b_vs_a
        ax.errorbar(value, yy, xerr=[[value - row.ci_low], [row.ci_high - value]],
                    fmt=markers[row.cohort], color=palette[row.cohort], ecolor=palette[row.cohort],
                    capsize=2, ms=4, label=row.cohort)
        ax.text(.99, yy + .13, f"q={row.exact_permutation_q:.2g}", transform=ax.get_yaxis_transform(),
                ha="right", va="center", fontsize=NOTE_SIZE_PT)
    handles, legend_labels = ax.get_legend_handles_labels()
    unique = dict(zip(legend_labels, handles))
    if len(unique) > 1:
        ax.legend(unique.values(), unique.keys(), frameon=False, fontsize=NOTE_SIZE_PT, ncol=2,
                  loc="upper right", borderaxespad=.1)
    ax.axvline(0, color="#555555", lw=.7)
    ax.set_xlim(-1.05, 1.05)
    ax.set_yticks([y_base[label] for label in labels], ["\n".join(textwrap.wrap(label, 35)) for label in labels],
                  fontsize=NOTE_SIZE_PT)
    ax.set_xlabel("Cliff's δ (NR − R)" if endpoint == "R/NR" else "Cliff's δ (NE − E)", fontsize=7)
    ax.set_title(f"{letter}  {title}", loc="left", fontsize=TITLE_SIZE_PT, fontweight="bold")
    ax.tick_params(axis="x", labelsize=TICK_SIZE_PT)
    for spine in ("top", "right", "left"):
        ax.spines[spine].set_visible(False)


def _format_activity_axis(ax, show_ylabel=True):
    """Show raw normalized percentages at the existing log2 coordinates."""
    ax.yaxis.set_major_locator(mpl.ticker.MaxNLocator(nbins=ACTIVITY_AXIS_TICKS, prune="both"))

    def percentage_label(log2_score, _position):
        percentage = 100.0 * np.exp2(log2_score)
        if percentage >= 0.01:
            return f"{percentage:.3g}"
        return f"{percentage:.1e}"

    ax.yaxis.set_major_formatter(mpl.ticker.FuncFormatter(percentage_label))
    ax.set_ylabel(ACTIVITY_AXIS_LABEL if show_ylabel else "", fontsize=7, labelpad=0)


def _box(ax, letter, patient, effects, association_id, endpoint_title):
    effect = effects.loc[effects.association_id.eq(association_id)]
    item = effect.iloc[0]
    cohorts = list(effect.cohort)
    arrays, positions, colors, tick_labels = [], [], [], []
    pos = 0.0
    rng = np.random.default_rng(RANDOM_SEED + ord(letter))
    for cohort in cohorts:
        block = patient.loc[
            patient.cohort.eq(cohort) & patient.cell_type.eq(item.cell_type) & patient.task.eq(item.task)
        ]
        short_cohort = cohort.replace("Bassez ", "B")
        endpoint_groups = ("R", "NR") if endpoint_title == "R/NR" else ("E", "NE")
        for (status, color), group_label in zip(
            (("Favorable", FAV_COLOR), ("Unfavorable", UNFAV_COLOR)), endpoint_groups
        ):
            values = block.loc[block.status.eq(status), "log2_median_score"].to_numpy()
            arrays.append(values); positions.append(pos); colors.append(color)
            tick_labels.append(group_label if cohort == "Shiao" else f"{short_cohort}\n{group_label}")
            ax.scatter(pos + rng.uniform(-.11, .11, len(values)), values, s=10, facecolor="white",
                       edgecolor=color, lw=.6, zorder=3)
            pos += 1
        pos += .65
    boxes = ax.boxplot(arrays, positions=positions, widths=.55, patch_artist=True, showfliers=False,
                       medianprops={"color": "#222222", "linewidth": .8},
                       whiskerprops={"linewidth": .65}, capprops={"linewidth": .65}, boxprops={"linewidth": .65})
    for box, color in zip(boxes["boxes"], colors):
        box.set_facecolor(color); box.set_alpha(.38)
    ax.set_xticks(positions, tick_labels, fontsize=TICK_SIZE_PT)
    _format_activity_axis(ax)
    task = "ERAD / recognition of misfolded protein" if item.task == ERAD else item.task
    ax.set_title(f"{letter}  {item.cell_type} · {endpoint_title}\n" + "\n".join(textwrap.wrap(task, 25)),
                 loc="left", fontsize=TITLE_SIZE_PT - 1.2, fontweight="bold", pad=3)
    stats = []
    for _, row in effect.iterrows():
        prefix = "" if len(effect) == 1 else row.cohort.replace("Bassez ", "B") + ": "
        stats.append(f"{prefix}δ={row.delta_group_b_vs_a:+.2f}, p={row.exact_permutation_p:.2g}, q={row.exact_permutation_q:.2g}")
    ax.text(.5, -.25, "\n".join(stats), transform=ax.transAxes, ha="center", fontsize=NOTE_SIZE_PT)
    cell_p = item.cell_wilcox_p
    cell_q = item.cell_wilcox_q
    p_text = "<1×10⁻³⁰⁰" if cell_p == 0 else f"{cell_p:.2g}"
    q_text = "<1×10⁻³⁰⁰" if cell_q == 0 else f"{cell_q:.2g}"
    ax.text(.5, -.36,
            f"Cell screen*: p={p_text}, q={q_text}",
            transform=ax.transAxes, ha="center", fontsize=NOTE_SIZE_PT, color="#555555")
    ax.tick_params(axis="y", labelsize=TICK_SIZE_PT)
    for spine in ("top", "right"):
        ax.spines[spine].set_visible(False)


def _format_adjusted_p(value):
    if value == 0 or value < 1e-300:
        return "<1e−300"
    if value >= 0.001:
        return f"{value:.3g}" + (" (ns)" if value >= 0.05 else "")
    return f"{value:.1e}".replace("e-", "e−")


def _add_de_bracket(ax, left, right, y, height, label):
    ax.plot([left, left, right, right], [y, y + height, y + height, y],
            color="#555555", lw=BRACKET_LINE_WIDTH, clip_on=False)
    ax.text((left + right) / 2, y + height, f"q={label}",
            ha="center", va="bottom", fontsize=NOTE_SIZE_PT, color="#444444")


def _treatment_box(ax, letter, patient, statistics, association_id):
    item = next(x for x in TREATMENT_PANELS if x["association_id"] == association_id)
    block = patient.loc[patient.association_id.eq(association_id)].copy()
    stat = statistics.loc[statistics.association_id.eq(association_id)].copy()
    positions = [0.0, 1.0, 2.0 + PAIR_GAP, 3.0 + PAIR_GAP]
    statuses = ("Favorable", "Unfavorable")
    colors = (FAV_COLOR, UNFAV_COLOR)
    arrays = []
    tick_labels = []
    point_records = []
    rng = np.random.default_rng(RANDOM_SEED + ord(letter))

    for group_index, (status, group_label, color) in enumerate(
        zip(statuses, item["group_labels"], colors)
    ):
        group = block.loc[block.status.eq(status)].copy()
        wide = group.pivot(index="patient", columns="timepoint_order", values="log2_median_score")
        if list(wide.columns) != [0, 1] or wide.isna().any().any():
            raise RuntimeError(f"Incomplete paired values for {association_id}/{group_label}")
        jitter = rng.uniform(-0.10, 0.10, len(wide))
        group_positions = positions[group_index * 2: group_index * 2 + 2]
        for timepoint_order, position in enumerate(group_positions):
            values = wide[timepoint_order].to_numpy()
            arrays.append(values)
            timepoint = group.loc[group.timepoint_order.eq(timepoint_order), "timepoint"].iloc[0]
            tick_labels.append(f"{group_label}\n{timepoint}")
            point_records.append((position, jitter, values, color))

    boxes = ax.boxplot(
        arrays, positions=positions, widths=BOX_WIDTH, patch_artist=True, showfliers=False,
        medianprops={"color": "#222222", "linewidth": 0.8},
        whiskerprops={"linewidth": 0.65}, capprops={"linewidth": 0.65},
        boxprops={"linewidth": 0.65},
    )
    for index, (box, color) in enumerate(zip(boxes["boxes"], colors * 2)):
        box.set_facecolor(colors[index // 2])
        box.set_alpha(PRE_BOX_ALPHA if index % 2 == 0 else ON_BOX_ALPHA)
    for position, jitter, values, color in point_records:
        ax.scatter(
            position + jitter, values, s=PATIENT_POINT_SIZE, facecolor="white",
            edgecolor=color, lw=PATIENT_POINT_EDGE_WIDTH, zorder=3,
        )

    values = block.log2_median_score.to_numpy()
    low, high = float(values.min()), float(values.max())
    span = max(high - low, 0.5)
    bracket_y = high + BRACKET_OFFSET_FRACTION * span
    bracket_height = BRACKET_HEIGHT_FRACTION * span
    for group_index, group_label in enumerate(item["group_labels"]):
        q_value = float(stat.loc[stat.group_label.eq(group_label), "de_adj_p_value"].iloc[0])
        left, right = positions[group_index * 2: group_index * 2 + 2]
        _add_de_bracket(ax, left, right, bracket_y, bracket_height, _format_adjusted_p(q_value))
    ax.set_ylim(low - 0.08 * span, bracket_y + 0.20 * span)

    ax.set_xticks(positions, tick_labels, fontsize=TICK_SIZE_PT)
    _format_activity_axis(ax)
    display_cell_type = "Tumor" if item["cell_type"] == "Tumor/epithelial" else item["cell_type"]
    ax.set_title(
        f"{letter}  {item['cohort']} · {display_cell_type}\n"
        + "\n".join(textwrap.wrap(f"{item['comparison']} · {item['display_task']}", 28)),
        loc="left", fontsize=TITLE_SIZE_PT - 1.2, fontweight="bold", pad=3,
    )
    interaction_p = float(stat.patient_interaction_p.iloc[0])
    ax.text(.5, -.27, f"Exact between-group Δ p={interaction_p:.2g}",
            transform=ax.transAxes, ha="center", fontsize=NOTE_SIZE_PT)
    ax.tick_params(axis="y", labelsize=TICK_SIZE_PT)
    for spine in ("top", "right"):
        ax.spines[spine].set_visible(False)
    ax.grid(False)


def _treatment_figure(paths):
    patient = pd.read_csv(paths["treatment_patient"])
    statistics = pd.read_csv(paths["treatment_statistics"])
    fig = plt.figure(figsize=(MAIN_WIDTH_MM * MM, TREATMENT_HEIGHT_MM * MM))
    grid = fig.add_gridspec(1, 4, left=.055, right=.992, bottom=.25, top=.78, wspace=.58)
    for axis, letter, item in zip(
        [fig.add_subplot(grid[0, index]) for index in range(4)], "ABCD", TREATMENT_PANELS,
    ):
        _treatment_box(axis, letter, patient, statistics, item["association_id"])
    fig.text(.5, .035,
             "Paired points are patient medians (≥20 cells at both timepoints) · q = DE BH-adjusted pooled-cell Wilcoxon p, descriptive only · Bassez E/NE denotes clonotype expansion",
             ha="center", fontsize=NOTE_SIZE_PT, color="#555555")
    return fig


def _patient_delta_table(patient, association_id):
    """Return one paired treatment-minus-baseline delta per eligible patient."""
    block = patient.loc[patient.association_id.eq(association_id)].copy()
    identity = ["patient", "status"]
    wide = block.pivot(index=identity, columns="timepoint_order", values="log2_median_score")
    if list(wide.columns) != [0, 1] or wide.isna().any().any():
        raise RuntimeError(f"Incomplete paired patient values for {association_id}")
    delta = wide.rename(columns={0: "baseline", 1: "treatment"}).reset_index()
    delta["delta_log2"] = delta.treatment - delta.baseline
    return delta


def _supplementary_patient_deltas(paths):
    """Patient-replicate concordance view for longitudinal panels I--L."""
    patient = pd.read_csv(paths["treatment_patient"])
    statistics = pd.read_csv(paths["treatment_statistics"])
    delta_tables = {
        item["association_id"]: _patient_delta_table(patient, item["association_id"])
        for item in TREATMENT_PANELS
    }
    max_abs = max(float(frame.delta_log2.abs().max()) for frame in delta_tables.values())
    axis_limit = np.ceil(max_abs / DELTA_AXIS_ROUNDING) * DELTA_AXIS_ROUNDING

    fig, axes = plt.subplots(
        1, 4, sharey=True, figsize=(MAIN_WIDTH_MM * MM, DELTA_HEIGHT_MM * MM),
    )
    fig.subplots_adjust(left=.065, right=.992, bottom=.26, top=.77, wspace=.32)
    for axis, letter, main_panel, item in zip(axes, "ABCD", "IJKL", TREATMENT_PANELS):
        association_id = item["association_id"]
        delta = delta_tables[association_id]
        stat = statistics.loc[statistics.association_id.eq(association_id)]
        rng = np.random.default_rng(RANDOM_SEED + ord(letter))
        for x_position, (status, group_label, color) in enumerate(zip(
            ("Favorable", "Unfavorable"), item["group_labels"], (FAV_COLOR, UNFAV_COLOR),
        )):
            values = delta.loc[delta.status.eq(status), "delta_log2"].to_numpy()
            expected_n = item["expected_patients"][x_position]
            if len(values) != expected_n:
                raise RuntimeError(
                    f"Unexpected patient count for {association_id}/{group_label}: "
                    f"{len(values)} != {expected_n}"
                )
            jitter = rng.uniform(-DELTA_JITTER_WIDTH, DELTA_JITTER_WIDTH, len(values))
            axis.scatter(
                x_position + jitter, values, s=DELTA_POINT_SIZE, facecolor="white",
                edgecolor=color, linewidth=DELTA_POINT_EDGE_WIDTH, zorder=3,
            )
            median = float(np.median(values))
            axis.plot(
                [x_position - DELTA_MEDIAN_HALF_WIDTH, x_position + DELTA_MEDIAN_HALF_WIDTH],
                [median, median], color=color, linewidth=DELTA_MEDIAN_LINE_WIDTH,
                solid_capstyle="round", zorder=4,
            )
            direction = 1 if median > 0 else -1
            concordant = int((np.sign(values) == direction).sum())
            arrow = "↑" if direction > 0 else "↓"
            axis.text(
                x_position, -.17, f"{concordant}/{len(values)} {arrow}",
                transform=axis.get_xaxis_transform(), ha="center", va="top",
                fontsize=NOTE_SIZE_PT, color=color,
            )

        axis.axhline(
            0, color=DELTA_ZERO_LINE_COLOR, linewidth=DELTA_ZERO_LINE_WIDTH,
            linestyle="--", zorder=1,
        )
        axis.set_xlim(-.52, 1.52)
        axis.set_ylim(-axis_limit, axis_limit)
        axis.set_xticks((0, 1), item["group_labels"], fontsize=TICK_SIZE_PT)
        axis.tick_params(axis="y", labelsize=TICK_SIZE_PT)
        display_cell_type = "Tumor" if item["cell_type"] == "Tumor/epithelial" else item["cell_type"]
        title = (
            f"{letter}  Main {main_panel} · {item['cohort']} · {display_cell_type}\n"
            + "\n".join(textwrap.wrap(f"{item['comparison']} · {item['display_task']}", 27))
        )
        axis.text(
            0, 1.17, title, transform=axis.transAxes, ha="left", va="bottom",
            fontsize=TITLE_SIZE_PT - 1.2, fontweight="bold", clip_on=False,
        )
        interaction_p = float(stat.patient_interaction_p.iloc[0])
        axis.text(
            0, 1.04, f"Exact between-group Δ p={interaction_p:.3g} (nominal)",
            transform=axis.transAxes, ha="left", va="bottom", fontsize=NOTE_SIZE_PT,
            color="#444444", clip_on=False,
        )
        for spine in ("top", "right"):
            axis.spines[spine].set_visible(False)
        axis.grid(False)
    axes[0].set_ylabel("Treatment − baseline\nΔ log2(patient-median Method 5 activity)",
                       fontsize=NOTE_SIZE_PT)
    fig.text(
        .5, .035,
        "One point per paired patient (≥20 target cells/timepoint) · colored bar = group median Δ · arrows report concordance with the median direction · exact two-sided Mann–Whitney test compares patient Δ between groups; nominal p, no multiplicity correction · Bassez E/NE denotes clonotype expansion",
        ha="center", fontsize=NOTE_SIZE_PT, color="#555555",
    )
    return fig


def _main(root, paths):
    v8=prepare_v8_data(root,force=False)
    patient=pd.read_csv(v8["eligible_patient_scores.csv.gz"])
    effects=pd.read_csv(paths["effects"])
    treatment_patient=pd.read_csv(paths["treatment_patient"])
    treatment_statistics=pd.read_csv(paths["treatment_statistics"])
    fig=plt.figure(figsize=(MAIN_WIDTH_MM*MM,165*MM))
    outer=fig.add_gridspec(3,1,height_ratios=(.82,.52,.68),hspace=.42,left=.055,right=.992,bottom=.06,top=.985)
    _upset(fig.add_subplot(outer[0]),root)
    baseline=outer[1].subgridspec(1,2,wspace=.38)
    for ax,spec in zip([fig.add_subplot(baseline[0,j]) for j in range(2)],
        [("E","RNR_TCELL_ERAD","R/NR"),("F","ENE_MYELOID_GLYCOGEN","E/NE")]):
        _box(ax,spec[0],patient,effects,spec[1],spec[2])
    longitudinal=outer[2].subgridspec(1,4,wspace=.58)
    for axis,letter,item in zip([fig.add_subplot(longitudinal[0,j]) for j in range(4)],"GHIJ",TREATMENT_PANELS):
        _treatment_box(axis,letter,treatment_patient,treatment_statistics,item["association_id"])
    fig.text(.5,.008,"Patient medians are the biological replicate · pooled-cell Wilcoxon/BH values are descriptive · Bassez E/NE denotes clonotype expansion, not clinical response",ha="center",fontsize=NOTE_SIZE_PT,color="#555555")
    return fig


def _save_full_canvas(fig, stem, close=True):
    """Save without tight cropping so the reserved lower composition area remains blank."""
    result = {}
    for extension in ("png", "pdf", "svg", "tiff"):
        path = stem.with_suffix("." + extension)
        fig.savefig(path, dpi=OUTPUT_DPI, facecolor="white")
        result[extension] = str(path)
    if close:
        plt.close(fig)
    return result


def _supplementary_evidence(paths):
    evidence = pd.read_csv(paths["evidence"])
    effects = pd.read_csv(paths["effects"])
    fig = plt.figure(figsize=(MAIN_WIDTH_MM * MM, 62 * MM))
    axis = fig.add_axes([.07, .22, .915, .68])
    _evidence_summary(axis, evidence, effects)
    fig.text(.5, .025,
             "Patient-level evidence from the frozen v8 screen · all BH q ≥ 0.05 (exploratory) · Bassez E/NE is not clinical response",
             ha="center", fontsize=NOTE_SIZE_PT, color="#555555")
    return fig


def _supplementary_unbiased_audit(root):
    data = prepare_v8_data(root, force=False)
    patient = pd.read_csv(data["eligible_patient_scores.csv.gz"])
    rnr_screen = pd.read_csv(data["rnr_full_screen.csv"])
    ene_screen = pd.read_csv(data["ene_full_screen.csv"])
    ranked_rnr = pd.read_csv(data["rnr_ranked_candidates.csv"])
    ranked_ene = pd.read_csv(data["ene_ranked_candidates.csv"])
    selected_rnr = pd.read_csv(data["rnr_selected_tasks.csv"])
    selected_ene = pd.read_csv(data["ene_selected_tasks.csv"])

    fig = plt.figure(figsize=(MAIN_WIDTH_MM * MM, AUDIT_HEIGHT_MM * MM))
    outer = fig.add_gridspec(3, 1, height_ratios=[.50, 1.24, 1.65], hspace=.48,
                             left=.06, right=.985, bottom=.06, top=.985)
    _upset(fig.add_subplot(outer[0]), root)
    selection = outer[1].subgridspec(1, 2, wspace=.42)
    _dotplot(fig.add_subplot(selection[0]), "B", "Clinical R/NR candidates", ranked_rnr,
             "abs_delta", ("R", "NR"))
    _dotplot(fig.add_subplot(selection[1]), "C", "Clonotype E/NE candidates", ranked_ene,
             "min_abs_delta", ("E", "NE"))
    boxes = outer[2].subgridspec(2, 3, hspace=.72, wspace=.32)
    for index, (letter, cell_type) in enumerate(zip("FGH", COMPARTMENTS)):
        task = selected_rnr.loc[selected_rnr.cell_type.eq(cell_type), "task"].iloc[0]
        axis = fig.add_subplot(boxes[0, index])
        _box_panel(axis, letter, cell_type, task, patient, ("Shiao",), ("R", "NR"),
                   rnr_screen, "R/NR")
        _format_activity_axis(axis)
    for index, (letter, cell_type) in enumerate(zip("IJK", COMPARTMENTS)):
        task = selected_ene.loc[selected_ene.cell_type.eq(cell_type), "task"].iloc[0]
        axis = fig.add_subplot(boxes[1, index])
        _box_panel(axis, letter, cell_type, task, patient, ("Bassez A", "Bassez B"),
                   ("E", "NE"), ene_screen, "E/NE")
        _format_activity_axis(axis)
    fig.text(.5, .018,
             "Points are patient medians · exact two-sided permutation tests · BH within endpoint screen · Bassez E/NE is not clinical response · logarithmic spacing; ticks show raw activity (%)",
             ha="center", fontsize=NOTE_SIZE_PT, color="#555555")
    return fig

def _clean_umap_axis(ax):
    ax.set_xticks([]); ax.set_yticks([])
    for spine in ax.spines.values():
        spine.set_visible(False)


def _draw_background(ax, background):
    ax.scatter(
        background.umap_1, background.umap_2, s=UMAP_BACKGROUND_SIZE,
        color=UMAP_BACKGROUND_COLOR, alpha=UMAP_BACKGROUND_ALPHA,
        linewidths=0, rasterized=UMAP_RASTERIZED,
    )


def _draw_baseline_map(ax, background, overlay, row, letter):
    bg = background.loc[
        background.cohort.eq(row.cohort) & background.condition.eq(row.condition)
    ]
    points = overlay.loc[overlay.map_id.eq(row.map_id)].sort_values("log10_score_plus_epsilon")
    norm = Normalize(row.color_low, row.color_high)
    _draw_background(ax, bg)
    ax.scatter(
        points.umap_1, points.umap_2, c=points.log10_score_plus_epsilon,
        cmap=UMAP_CMAP, norm=norm, s=UMAP_OVERLAY_SIZE, alpha=UMAP_OVERLAY_ALPHA,
        linewidths=0, rasterized=UMAP_RASTERIZED,
    )
    task_title = "\n".join(textwrap.wrap(f"{row.condition} · {row.display_task}", 31))
    ax.set_title(
        f"{letter}  {row.main_panel} · {row.cohort} · {row.cell_type}\n{task_title}",
        loc="left", fontsize=UMAP_TITLE_SIZE_PT, fontweight="bold", pad=3,
    )
    ax.text(
        .5, -.08, f"{row.patient_count_label} · {int(row.n_overlay_cells):,} colored cells",
        transform=ax.transAxes, ha="center", va="top", fontsize=UMAP_NOTE_SIZE_PT,
    )
    _clean_umap_axis(ax)
    return ScalarMappable(norm=norm, cmap=UMAP_CMAP)


def supplementary_baseline_umaps(paths: Mapping[str, Path]):
    background = pd.read_parquet(paths["background"])
    overlay = pd.read_parquet(paths["overlay"])
    metadata = pd.read_csv(paths["metadata"]).query("figure_kind == 'baseline'").sort_values("display_order")
    if len(metadata) != 6:
        raise RuntimeError("Expected six baseline task-UMAP maps")
    fig, axes = plt.subplots(2, 3, figsize=(180 * MM, UMAP_BASELINE_HEIGHT_MM * MM))
    fig.subplots_adjust(left=.035, right=.985, bottom=.12, top=.94, wspace=.24, hspace=.48)
    for ax, letter, row in zip(axes.flat, "ABCDEF", metadata.itertuples(index=False)):
        sm = _draw_baseline_map(ax, background, overlay, row, letter)
        cb = fig.colorbar(sm, ax=ax, orientation="horizontal", fraction=.035, pad=.13, aspect=25)
        cb.set_label("log10(score + 1e−10)", fontsize=UMAP_COLORBAR_SIZE_PT, labelpad=1)
        cb.ax.tick_params(labelsize=UMAP_COLORBAR_SIZE_PT, length=2)
    fig.text(
        .5, .018,
        "Frozen embeddings · grey = all endpoint-annotated cells in condition · color = eligible target-cell activity · descriptive localization only; compare colors within association",
        ha="center", fontsize=UMAP_NOTE_SIZE_PT, color="#555555",
    )
    return fig


def _draw_activity_facet(ax, background, points, row, status, group_label, condition, title):
    bg = background.loc[
        background.cohort.eq(row.cohort) & background.condition.eq(condition)
    ]
    selected = points.loc[
        points.condition.eq(condition) & points.status.eq(status)
    ].sort_values("log10_score_plus_epsilon")
    if selected.empty:
        raise RuntimeError(f"Empty longitudinal facet: {row.association_id}/{group_label}/{condition}")
    norm = Normalize(row.color_low, row.color_high)
    _draw_background(ax, bg)
    ax.scatter(
        selected.umap_1, selected.umap_2, c=selected.log10_score_plus_epsilon,
        cmap=UMAP_CMAP, norm=norm, s=UMAP_OVERLAY_SIZE, alpha=UMAP_OVERLAY_ALPHA,
        linewidths=0, rasterized=UMAP_RASTERIZED,
    )
    ax.set_title(title, loc="left", fontsize=UMAP_TITLE_SIZE_PT, fontweight="bold", pad=2)
    ax.text(
        .5, -.07,
        f"n={selected.patient.nunique()} patients · {len(selected):,} cells",
        transform=ax.transAxes, ha="center", va="top", fontsize=UMAP_NOTE_SIZE_PT,
    )
    _clean_umap_axis(ax)
    return ScalarMappable(norm=norm, cmap=UMAP_CMAP)


def _draw_categorical_facet(ax, background, points, row, labels):
    conditions = sorted(points.condition.unique(), key=lambda value: 0 if value in {"Base", "Pre"} else 1)
    bg = background.loc[
        background.cohort.eq(row.cohort) & background.condition.isin(conditions)
    ]
    if points.cell_id.duplicated().any():
        raise RuntimeError(f"Duplicated cells in categorical facet: {row.association_id}")
    _draw_background(ax, bg)
    for status, label, color in zip(
        ("Favorable", "Unfavorable"), labels, (FAV_COLOR, UNFAV_COLOR),
    ):
        selected = points.loc[points.status.eq(status)]
        ax.scatter(
            selected.umap_1, selected.umap_2, s=UMAP_OVERLAY_SIZE,
            color=color, alpha=UMAP_OVERLAY_ALPHA, linewidths=0,
            rasterized=UMAP_RASTERIZED,
        )
    ax.set_title(
        f"{labels[0]} vs {labels[1]} · both timepoints",
        loc="left", fontsize=UMAP_TITLE_SIZE_PT, fontweight="bold", pad=2,
    )
    handles = [
        Line2D([0], [0], marker="o", linestyle="", markersize=3.4, color=color, label=label)
        for label, color in zip(labels, (FAV_COLOR, UNFAV_COLOR))
    ]
    ax.legend(handles=handles, loc="lower center", ncol=2, frameon=False,
              fontsize=UMAP_NOTE_SIZE_PT, handletextpad=.25, columnspacing=.7)
    counts = []
    for status, label in zip(("Favorable", "Unfavorable"), labels):
        selected = points.loc[points.status.eq(status)]
        counts.append(f"{label}: n={selected.patient.nunique()}, {len(selected):,} cells")
    ax.text(.5, -.07, " · ".join(counts), transform=ax.transAxes,
            ha="center", va="top", fontsize=UMAP_NOTE_SIZE_PT)
    _clean_umap_axis(ax)


def supplementary_longitudinal_umaps(paths: Mapping[str, Path]):
    background = pd.read_parquet(paths["background"])
    overlay = pd.read_parquet(paths["overlay"])
    metadata = pd.read_csv(paths["metadata"]).query("figure_kind == 'longitudinal'")
    panels = list("IJKL")
    if len(metadata) != 8 or sorted(metadata.main_panel.unique()) != panels:
        raise RuntimeError("Expected two staged conditions for each longitudinal panel I-L")
    fig, axes = plt.subplots(4, 5, figsize=(180 * MM, UMAP_LONGITUDINAL_HEIGHT_MM * MM))
    fig.subplots_adjust(left=.035, right=.992, bottom=.045, top=.965, wspace=.13, hspace=.70)
    for row_index, (letter, panel) in enumerate(zip("ABCD", panels)):
        group_metadata = metadata.loc[metadata.main_panel.eq(panel)].sort_values("condition_order")
        baseline_row, treatment_row = tuple(group_metadata.itertuples(index=False))
        points = overlay.loc[overlay.association_id.eq(baseline_row.association_id)].copy()
        labels = ("R", "NR") if baseline_row.cohort == "Shiao" else ("E", "NE")
        conditions = (baseline_row.condition, treatment_row.condition)
        row_title = f"{letter}  {panel} · {baseline_row.cohort} · {baseline_row.cell_type} · {baseline_row.display_task}"
        axes[row_index, 0].text(
            0, 1.16, row_title, transform=axes[row_index, 0].transAxes,
            fontsize=UMAP_TITLE_SIZE_PT, fontweight="bold", ha="left", va="bottom",
            clip_on=False,
        )
        sm = None
        for column, (status, group_label, condition) in enumerate((
            ("Favorable", labels[0], conditions[0]),
            ("Favorable", labels[0], conditions[1]),
            ("Unfavorable", labels[1], conditions[0]),
            ("Unfavorable", labels[1], conditions[1]),
        )):
            sm = _draw_activity_facet(
                axes[row_index, column], background, points, baseline_row,
                status, group_label, condition, f"{group_label} · {condition}",
            )
        _draw_categorical_facet(axes[row_index, 4], background, points, baseline_row, labels)
        cb = fig.colorbar(
            sm, ax=axes[row_index, :4].tolist(), orientation="horizontal",
            fraction=.018, pad=.085, aspect=58,
        )
        cb.set_label("log10(score + 1e−10) · shared across four activity maps", fontsize=UMAP_COLORBAR_SIZE_PT, labelpad=1)
        cb.ax.tick_params(labelsize=UMAP_COLORBAR_SIZE_PT, length=2)
    fig.text(
        .5, .009,
        "Frozen embeddings · activity facets use eligible target cells from one group/timepoint · fifth map pools both retained timepoints and colors target cells by E/NE or R/NR · descriptive only",
        ha="center", fontsize=UMAP_NOTE_SIZE_PT, color="#555555",
    )
    return fig


def renderer_metadata() -> dict:
    return {
        "python": platform.python_version(), "matplotlib": mpl.__version__,
        "cnsplots": "unavailable; established matplotlib fallback",
        "umap_coordinates": "frozen X_umap; no embedding recomputed",
        "activity_transformation": DISPLAY_TRANSFORMATION,
        "activity_colormap": UMAP_CMAP,
        "display_epsilon": EPSILON,
        "color_clipping": "association-specific q02-q98 shared across four group-by-timepoint activity facets",
        "longitudinal_layout": "four tasks x five maps; fifth categorical map pools both retained timepoints",
    }

def supplementary_unbiased_audit_compact(root):
    data = prepare_v8_data(root, force=False)
    patient = pd.read_csv(data["eligible_patient_scores.csv.gz"])
    rnr_screen = pd.read_csv(data["rnr_full_screen.csv"]); ene_screen = pd.read_csv(data["ene_full_screen.csv"])
    ranked_rnr = pd.read_csv(data["rnr_ranked_candidates.csv"]); ranked_ene = pd.read_csv(data["ene_ranked_candidates.csv"])
    selected_rnr = pd.read_csv(data["rnr_selected_tasks.csv"]); selected_ene = pd.read_csv(data["ene_selected_tasks.csv"])
    fig = plt.figure(figsize=(MAIN_WIDTH_MM * MM, 188 * MM))
    outer = fig.add_gridspec(3, 1, height_ratios=[0.82, 1, 1], hspace=.63, left=.065, right=.985, bottom=.07, top=.97)
    top = outer[0].subgridspec(1, 2, wspace=.42)
    _dotplot(fig.add_subplot(top[0]), "A", "Clinical R/NR candidates", ranked_rnr, "abs_delta", ("R", "NR"))
    _dotplot(fig.add_subplot(top[1]), "B", "Clonotype E/NE candidates", ranked_ene, "min_abs_delta", ("E", "NE"))
    row_specs = (("CDE", selected_rnr, rnr_screen, ("Shiao",), ("R","NR"), "R/NR"), ("FGH", selected_ene, ene_screen, ("Bassez A","Bassez B"), ("E","NE"), "E/NE"))
    for row_index, (letters, selected, screen, cohorts, labels, endpoint) in enumerate(row_specs, start=1):
        grid = outer[row_index].subgridspec(1, 3, wspace=.35)
        for index, (letter, cell_type) in enumerate(zip(letters, COMPARTMENTS)):
            task = selected.loc[selected.cell_type.eq(cell_type), "task"].iloc[0]
            axis = fig.add_subplot(grid[index]); _box_panel(axis, letter, cell_type, task, patient, cohorts, labels, screen, endpoint); _format_activity_axis(axis)
    fig.text(.5, .018, "Complete endpoint-specific audit · one median per patient · exact two-sided permutation tests · BH within endpoint screen · Bassez E/NE denotes clonotype expansion", ha="center", fontsize=NOTE_SIZE_PT, color="#555555")
    return fig


## Individual main-figure plots

These seven source figures use the same plotting primitives and frozen tables as the compiled figure below.

In [ ]:
def render_main_method_overlap():
    fig, ax = plt.subplots(figsize=(MAIN_WIDTH_MM * MM, 68 * MM))
    fig.subplots_adjust(left=.055, right=.992, bottom=.12, top=.96)
    _upset(ax, ROOT)
    return fig


def render_main_baseline(letter, association_id, endpoint_title):
    patient_frame = pd.read_csv(DATA_DIR / "eligible_patient_scores.csv.gz")
    effects_frame = pd.read_csv(PLOT_PATHS["effects"])
    fig, ax = plt.subplots(figsize=(86 * MM, 62 * MM))
    fig.subplots_adjust(left=.20, right=.97, bottom=.20, top=.84)
    _box(ax, letter, patient_frame, effects_frame, association_id, endpoint_title)
    return fig


def render_main_longitudinal(letter, association_id):
    patient_frame = pd.read_csv(PLOT_PATHS["treatment_patient"])
    statistics_frame = pd.read_csv(PLOT_PATHS["treatment_statistics"])
    fig, ax = plt.subplots(figsize=(44 * MM, 68 * MM))
    fig.subplots_adjust(left=.25, right=.97, bottom=.18, top=.82)
    _treatment_box(ax, letter, patient_frame, statistics_frame, association_id)
    return fig


main_component_figures = {
    "main_component_A-D_method_overlap": render_main_method_overlap(),
    "main_component_E_shiao_tcell_erad": render_main_baseline("E", "RNR_TCELL_ERAD", "R/NR"),
    "main_component_F_bassez_myeloid_glycogen": render_main_baseline("F", "ENE_MYELOID_GLYCOGEN", "E/NE"),
}
for letter, item in zip("GHIJ", TREATMENT_PANELS):
    main_component_figures[f"main_component_{letter}_{item['association_id'].lower()}"] = (
        render_main_longitudinal(letter, item["association_id"])
    )

main_component_outputs = {}
for name, figure in main_component_figures.items():
    display(figure)
    main_component_outputs[name] = _save_v8(figure, OUTPUT_DIR / f"Figure_3_version_2_{name}")

display(pd.DataFrame({"main component": list(main_component_outputs)}))

## Compiled Main Figure 3

The compilation is deliberately last and retains the established A–J layout.

In [ ]:
main_figure = _main(ROOT, PLOT_PATHS)
display(main_figure)
main_outputs = _save_full_canvas(main_figure, OUTPUT_DIR / "Figure_3_version_2_main")


# Part II — Supplementary Figures

Prepare supplementary-only inputs, show each source figure independently, and then compile the three final supplementary pages.

## Supplementary inputs and renderers

### Bassez cell-level GO differential-activity heatmaps

Raw scKNIFE GO-node activities were tested using the established cell-level metabolic-task procedure: Scanpy Wilcoxon On versus Pre, separately within E and NE and each cell type. BH correction is global within each cohort × expansion group. These pooled-cell results are descriptive benchmarking rather than patient-level discovery.


In [ ]:
BASSEZ_GO_RESULTS_PATH = DATA_DIR / "bassez_go_cell_de_results.csv.gz"
BASSEZ_GO_AUDIT_PATH = DATA_DIR / "bassez_go_cell_de_audit.json"
bassez_go_cell_results = pd.read_csv(BASSEZ_GO_RESULTS_PATH)
bassez_go_audit = json.loads(BASSEZ_GO_AUDIT_PATH.read_text())
assert bassez_go_audit["go_universe"] == 6036
assert set(bassez_go_cell_results.cohort) == {"Bassez A", "Bassez B"}
display(pd.DataFrame({
    "cohort": ["Bassez A", "Bassez B"],
    "DE rows": [
        int(bassez_go_cell_results.cohort.eq("Bassez A").sum()),
        int(bassez_go_cell_results.cohort.eq("Bassez B").sum()),
    ],
}))


In [ ]:
# Bassez clonotype-expansion metabolic-task heatmap renderer.
# Plotting transformations only: completed DE values are not recomputed.
import seaborn as sns
from matplotlib.backends.backend_pdf import PdfPages

BASSEZ_HEATMAP_INPUTS = {
    "A": DATA_DIR / "de/bassez_A_knife.csv",
    "B": DATA_DIR / "de/bassez_B_knife.csv",
}
BASSEZ_HEATMAP_TOP_N = 50
BASSEZ_HEATMAP_PADJ = 0.05
BASSEZ_HEATMAP_PAGE_SIZE = (A4_PORTRAIT_WIDTH_MM * MM, A4_PORTRAIT_HEIGHT_MM * MM)
BASSEZ_HEATMAP_TASK_FONT_PT = 4.0
BASSEZ_HEATMAP_COLUMN_FONT_PT = 5.0
BASSEZ_HEATMAP_TITLE_PT = 9.0

def prepare_top_e_ne_heatmap(de_combined, top_n=BASSEZ_HEATMAP_TOP_N,
                             pval_cutoff=BASSEZ_HEATMAP_PADJ):
    """Faithful data-selection logic from scknife.plotting.plot_top_e_ne_heatmap."""
    frame = de_combined.copy()
    required = {"feature", "expansion_group", "cell_type", "log2FC", "adj_p_value"}
    missing = required.difference(frame.columns)
    if missing:
        raise ValueError(f"Heatmap DE table lacks columns: {sorted(missing)}")
    frame["label"] = frame["expansion_group"] + "_" + frame["cell_type"]
    significant = frame.loc[frame["adj_p_value"] < pval_cutoff].copy()
    if significant.empty:
        raise RuntimeError("No adjusted-p significant tasks for E/NE heatmap")
    feature_group = significant.groupby(["feature", "expansion_group"])["log2FC"].mean()
    top_e = feature_group.xs("E", level="expansion_group").nlargest(top_n).index
    top_ne = feature_group.xs("NE", level="expansion_group").nlargest(top_n).index
    selected_slots = list(top_e) + list(top_ne)
    pivot = (
        frame.loc[frame["feature"].isin(selected_slots)]
        .pivot_table(index="feature", columns="label", values="log2FC", aggfunc="first")
        .fillna(0)
        .reindex(selected_slots)
    )
    audit = {
        "input_rows": int(len(frame)), "significant_rows": int(len(significant)),
        "top_n_per_group": int(top_n), "selected_slots": int(len(selected_slots)),
        "unique_selected_tasks": int(len(set(selected_slots))),
        "matrix_rows": int(pivot.shape[0]), "matrix_columns": int(pivot.shape[1]),
        "columns": pivot.columns.astype(str).tolist(),
        "e_tasks": list(map(str, top_e)),
        "ne_tasks": list(map(str, top_ne)),
    }
    return pivot, audit

def render_top_e_ne_heatmap(de_combined, panel_label, cohort_title):
    """Original Seaborn clustermap renderer adapted to a full A4 portrait page."""
    pivot, audit = prepare_top_e_ne_heatmap(de_combined)
    grid = sns.clustermap(
        pivot, cmap="RdBu_r", center=0, xticklabels=True, yticklabels=True,
        cbar_kws={"label": "log2FC"}, row_cluster=True, col_cluster=True,
        figsize=BASSEZ_HEATMAP_PAGE_SIZE,
        dendrogram_ratio=(.13, .15), colors_ratio=.01,
        cbar_pos=(.018, .78, .035, .15),
    )
    # Reserve the right half of the portrait page for complete metabolic-task labels.
    grid.ax_row_dendrogram.set_position([.045, .105, .075, .675])
    grid.ax_heatmap.set_position([.12, .105, .285, .675])
    grid.ax_col_dendrogram.set_position([.12, .79, .285, .145])
    grid.cax.set_position([.018, .79, .035, .145])
    grid.ax_heatmap.tick_params(axis="y", labelsize=BASSEZ_HEATMAP_TASK_FONT_PT, length=1.5, pad=1)
    grid.ax_heatmap.tick_params(axis="x", labelsize=BASSEZ_HEATMAP_COLUMN_FONT_PT, length=1.5, pad=1)
    grid.ax_heatmap.set_xlabel("Cell type", fontsize=AXIS_TITLE_SIZE_PT)
    grid.ax_heatmap.set_ylabel("Metabolic tasks", fontsize=AXIS_TITLE_SIZE_PT, labelpad=8)
    grid.cax.tick_params(labelsize=TICK_SIZE_PT)
    grid.cax.yaxis.label.set_size(AXIS_TITLE_SIZE_PT)
    grid.fig.suptitle(
        f"{panel_label}  {cohort_title}: top E/NE metabolic tasks",
        x=.045, y=.975, ha="left", fontsize=BASSEZ_HEATMAP_TITLE_PT, fontweight="bold",
    )
    return grid, pivot, audit

def save_heatmap_page(grid, stem):
    outputs = {}
    for extension in ("png", "pdf", "svg", "tiff"):
        path = stem.with_suffix("." + extension)
        grid.fig.savefig(path, format=extension, dpi=OUTPUT_DPI, facecolor="white")
        outputs[extension] = str(path)
    return outputs

for cohort, path in BASSEZ_HEATMAP_INPUTS.items():
    if not path.is_file():
        raise FileNotFoundError(path)
    print(f"Bassez {cohort} heatmap source: {path.relative_to(ROOT)}")
    print("columns: feature, expansion_group, cell_type, log2FC, adj_p_value")


In [ ]:
def supplementary_audit_ab(root):
    data = prepare_v8_data(root, force=False)
    ranked_rnr = pd.read_csv(data["rnr_ranked_candidates.csv"])
    ranked_ene = pd.read_csv(data["ene_ranked_candidates.csv"])
    fig, axes = plt.subplots(1, 2, figsize=(180 * MM, 68 * MM))
    fig.subplots_adjust(left=S3A_CANDIDATE_LEFT, right=.985, bottom=.18, top=.88, wspace=.42)
    _dotplot(axes[0], "A", "Clinical R/NR candidates", ranked_rnr, "abs_delta", ("R", "NR"))
    _dotplot(axes[1], "B", "Clonotype E/NE candidates", ranked_ene, "min_abs_delta", ("E", "NE"))
    fig.suptitle("Audit A–B  Unbiased endpoint-specific candidate screens", x=.01, ha="left", weight="bold", fontsize=TITLE_SIZE_PT)
    return fig


def supplementary_audit_ch(root):
    data = prepare_v8_data(root, force=False)
    patient = pd.read_csv(data["eligible_patient_scores.csv.gz"])
    rnr_screen = pd.read_csv(data["rnr_full_screen.csv"])
    ene_screen = pd.read_csv(data["ene_full_screen.csv"])
    selected_rnr = pd.read_csv(data["rnr_selected_tasks.csv"])
    selected_ene = pd.read_csv(data["ene_selected_tasks.csv"])
    fig, axes = plt.subplots(
        1, 6, figsize=(S3A_SOURCE_BOX_WIDTH_MM * MM, S3A_SOURCE_BOX_HEIGHT_MM * MM),
    )
    fig.subplots_adjust(left=.04, right=.992, bottom=.25, top=.76, wspace=.52)
    specs = (
        ("CDE", selected_rnr, rnr_screen, ("Shiao",), ("R", "NR"), "R/NR"),
        ("FGH", selected_ene, ene_screen, ("Bassez A", "Bassez B"), ("E", "NE"), "E/NE"),
    )
    panel_index = 0
    for letters, selected, screen, cohorts, labels, endpoint in specs:
        for letter, cell_type in zip(letters, COMPARTMENTS):
            task = selected.loc[selected.cell_type.eq(cell_type), "task"].iloc[0]
            _box_panel(
                axes[panel_index], letter, cell_type, task, patient, cohorts, labels,
                screen, endpoint, compact=True, show_ylabel=panel_index in (0, 3),
            )
            _format_activity_axis(axes[panel_index], show_ylabel=panel_index in (0, 3))
            panel_index += 1
    fig.suptitle(
        "Audit C–H  Patient-level selected-task distributions",
        x=.01, ha="left", weight="bold", fontsize=TITLE_SIZE_PT,
    )
    return fig


In [ ]:
from extractor_scripts.native_figure3_version_2_compositor import (
    compose_s3_a, compose_s3_b, compose_s3_c,
    prepare_bassez_go_heatmap, render_bassez_go_heatmap,
)
from extractor_scripts.mixed_vector_compositor import inspect_svg


## Individual supplementary plots

The source plots are presented in S3_A, S3_B, and S3_C narrative order before any page is compiled.

### S3_A source plots — candidate selection and patient-level audit

In [ ]:
supplementary_component_figures = {
    "supp_component_S3A_candidate_rankings": supplementary_audit_ab(ROOT),
    "supp_component_S3A_patient_distributions": supplementary_audit_ch(ROOT),
}
supplementary_component_outputs = {}
for name, figure in supplementary_component_figures.items():
    display(figure)
    supplementary_component_outputs[name] = _save_v8(
        figure, OUTPUT_DIR / f"Figure_3_version_2_{name}"
    )

### S3_B top row — Bassez A paired metabolic-task and GO heatmaps


In [ ]:
bassez_A_de = pd.read_csv(BASSEZ_HEATMAP_INPUTS["A"])
bassez_A_grid, BASSEZ_A_HEATMAP_MATRIX, BASSEZ_A_HEATMAP_AUDIT = render_top_e_ne_heatmap(
    bassez_A_de, "A", "Bassez A",
)
display(bassez_A_grid.fig)
supplementary_component_outputs["supp_component_S3B_bassez_A_metabolic"] = save_heatmap_page(
    bassez_A_grid, OUTPUT_DIR / "Figure_3_version_2_supp_component_S3B_bassez_A_metabolic",
)

bassez_go_A_grid, BASSEZ_GO_A_HEATMAP_MATRIX, BASSEZ_GO_A_HEATMAP_AUDIT = render_bassez_go_heatmap(
    globals(), "Bassez A", "B",
)
display(bassez_go_A_grid.fig)
supplementary_component_outputs["supp_component_S3B_bassez_A_GO_heatmap"] = save_heatmap_page(
    bassez_go_A_grid, OUTPUT_DIR / "Figure_3_version_2_supp_component_S3B_bassez_A_GO_heatmap",
)
display(pd.Series(BASSEZ_A_HEATMAP_AUDIT, name="Bassez A metabolic audit"))
display(pd.Series(BASSEZ_GO_A_HEATMAP_AUDIT, name="Bassez A GO audit"))


### S3_B bottom row — Bassez B paired metabolic-task and GO heatmaps


In [ ]:
bassez_B_de = pd.read_csv(BASSEZ_HEATMAP_INPUTS["B"])
bassez_B_grid, BASSEZ_B_HEATMAP_MATRIX, BASSEZ_B_HEATMAP_AUDIT = render_top_e_ne_heatmap(
    bassez_B_de, "C", "Bassez B",
)
display(bassez_B_grid.fig)
supplementary_component_outputs["supp_component_S3B_bassez_B_metabolic"] = save_heatmap_page(
    bassez_B_grid, OUTPUT_DIR / "Figure_3_version_2_supp_component_S3B_bassez_B_metabolic",
)

bassez_go_B_grid, BASSEZ_GO_B_HEATMAP_MATRIX, BASSEZ_GO_B_HEATMAP_AUDIT = render_bassez_go_heatmap(
    globals(), "Bassez B", "D",
)
display(bassez_go_B_grid.fig)
supplementary_component_outputs["supp_component_S3B_bassez_B_GO_heatmap"] = save_heatmap_page(
    bassez_go_B_grid, OUTPUT_DIR / "Figure_3_version_2_supp_component_S3B_bassez_B_GO_heatmap",
)

go_heatmap_component_audits = {
    "Bassez A": BASSEZ_GO_A_HEATMAP_AUDIT,
    "Bassez B": BASSEZ_GO_B_HEATMAP_AUDIT,
}
assert BASSEZ_GO_A_HEATMAP_AUDIT["selected_slots"] == 50
assert BASSEZ_GO_B_HEATMAP_AUDIT["selected_slots"] == 50
assert BASSEZ_GO_A_HEATMAP_AUDIT["unique_selected_terms"] == 50
assert BASSEZ_GO_B_HEATMAP_AUDIT["unique_selected_terms"] == 48
assert BASSEZ_GO_A_HEATMAP_MATRIX.shape == (50, 12)
assert BASSEZ_GO_B_HEATMAP_MATRIX.shape == (50, 11)
for cohort, expected in (("Bassez A", BASSEZ_GO_A_HEATMAP_AUDIT), ("Bassez B", BASSEZ_GO_B_HEATMAP_AUDIT)):
    repeated_matrix, repeated_audit = prepare_bassez_go_heatmap(globals(), cohort)
    assert repeated_audit["e_terms"] == expected["e_terms"]
    assert repeated_audit["ne_terms"] == expected["ne_terms"]
    pd.testing.assert_frame_equal(
        repeated_matrix,
        BASSEZ_GO_A_HEATMAP_MATRIX if cohort == "Bassez A" else BASSEZ_GO_B_HEATMAP_MATRIX,
    )
display(pd.Series(BASSEZ_B_HEATMAP_AUDIT, name="Bassez B metabolic audit"))
display(pd.Series(BASSEZ_GO_B_HEATMAP_AUDIT, name="Bassez B GO audit"))


### S3_C source plots — baseline and longitudinal UMAP localization

In [ ]:
baseline_umap_figure = supplementary_baseline_umaps(UMAP_PATHS)
display(baseline_umap_figure)
supplementary_component_outputs["supp_component_S3C_baseline_UMAP"] = _save_v8(
    baseline_umap_figure, OUTPUT_DIR / "Figure_3_version_2_supp_component_S3C_baseline_UMAP",
)

longitudinal_umap_figure = supplementary_longitudinal_umaps(UMAP_PATHS)
display(longitudinal_umap_figure)
supplementary_component_outputs["supp_component_S3C_longitudinal_UMAP"] = _save_v8(
    longitudinal_umap_figure, OUTPUT_DIR / "Figure_3_version_2_supp_component_S3C_longitudinal_UMAP",
)

assert len(supplementary_component_outputs) == 8
display(pd.DataFrame({"supplementary component": list(supplementary_component_outputs)}))

## Compiled supplementary figures

Only after the eight source figures are shown are they assembled into the three official pages.

In [ ]:
heatmap_matrix_snapshots = {
    "Bassez A metabolic": BASSEZ_A_HEATMAP_MATRIX.copy(deep=True),
    "Bassez A GO": BASSEZ_GO_A_HEATMAP_MATRIX.copy(deep=True),
    "Bassez B metabolic": BASSEZ_B_HEATMAP_MATRIX.copy(deep=True),
    "Bassez B GO": BASSEZ_GO_B_HEATMAP_MATRIX.copy(deep=True),
}
final_s3_figures = {
    "S3_A_final": compose_s3_a(globals()),
    "S3_B_final": compose_s3_b(globals()),
    "S3_C_final": compose_s3_c(globals()),
}
for label, matrix in (
    ("Bassez A metabolic", BASSEZ_A_HEATMAP_MATRIX),
    ("Bassez A GO", BASSEZ_GO_A_HEATMAP_MATRIX),
    ("Bassez B metabolic", BASSEZ_B_HEATMAP_MATRIX),
    ("Bassez B GO", BASSEZ_GO_B_HEATMAP_MATRIX),
):
    pd.testing.assert_frame_equal(matrix, heatmap_matrix_snapshots[label])

final_s3_outputs = {}
for name, figure in final_s3_figures.items():
    display(figure)
    final_s3_outputs[name] = _save_full_canvas(
        figure, OUTPUT_DIR / f"Figure_3_version_2_{name}"
    )
final_s3_outputs


## Final numerical, provenance, and output audit

In [ ]:
import subprocess
from matplotlib.backends.backend_pdf import PdfPages

assert len(main_component_outputs) == 7
assert len(supplementary_component_outputs) == 8
assert tuple(final_s3_outputs) == ("S3_A_final", "S3_B_final", "S3_C_final")

all_export_groups = {
    **main_component_outputs,
    **supplementary_component_outputs,
    "main": main_outputs,
    **final_s3_outputs,
}
for formats in all_export_groups.values():
    assert set(formats) == {"png", "pdf", "svg", "tiff"}
    for path in formats.values():
        assert Path(path).is_file() and Path(path).stat().st_size > 0

mixed_media_audit = {}
for name, outputs in final_s3_outputs.items():
    svg = inspect_svg(Path(outputs["svg"]))
    pdf_text = subprocess.run(
        ["pdftotext", outputs["pdf"], "-"], check=True, capture_output=True, text=True
    ).stdout
    fonts = subprocess.run(
        ["pdffonts", outputs["pdf"]], check=True, capture_output=True, text=True
    ).stdout
    assert svg["text_elements"] > 20 and len(pdf_text.strip()) > 500 and "TrueType" in fonts
    if name == "S3_B_final":
        assert "Metabolic tasks" in pdf_text
        assert "GO biological processes" in pdf_text
        assert "B-cell NE not tested" in pdf_text
    mixed_media_audit[name] = {
        **svg,
        "pdf_text_characters": len(pdf_text.strip()),
        "pdf_embedded_truetype": True,
    }

assert mixed_media_audit["S3_A_final"]["image_elements"] == 0
assert mixed_media_audit["S3_B_final"]["image_elements"] == 0
assert mixed_media_audit["S3_C_final"]["image_elements"] > 0

three_page_pdf = OUTPUT_DIR / "Figure_3_version_2_supplementary_three_page.pdf"
with PdfPages(three_page_pdf) as pdf:
    for name in ("S3_A_final", "S3_B_final", "S3_C_final"):
        pdf.savefig(final_s3_figures[name], facecolor="white")
page_count = subprocess.run(
    ["pdfinfo", three_page_pdf], check=True, capture_output=True, text=True
).stdout
assert "Pages:           3" in page_count

matrix_hashes = {
    label: hashlib.sha256(matrix.to_csv().encode()).hexdigest()
    for label, matrix in heatmap_matrix_snapshots.items()
}
manifest_path = OUTPUT_DIR / "Figure_3_version_2_manifest.json"
manifest_update = {
    "revision": "figure3_version_2_s3a_single_row_boxplots",
    "notebook_structure": {
        "part_1": "individual main plots followed by compiled main",
        "part_2": "eight individual supplementary plots followed by S3_A/S3_B/S3_C compilations",
    },
    "main_layout": {
        "A-D": "method overlap",
        "E": "Shiao R/NR T-cell ERAD",
        "F": "Bassez E/NE myeloid glycogen",
        "G-J": "four longitudinal examples",
    },
    "supplementary_layout": {
        "S3_A_final": ["candidate rankings", "six slim patient-distribution boxplots in one row"],
        "S3_B_final": [
            "top row: Bassez A metabolic-task and GO heatmaps",
            "bottom row: Bassez B metabolic-task and GO heatmaps",
        ],
        "S3_C_final": ["baseline UMAPs", "longitudinal UMAPs"],
    },
    "integrated_heatmap_layout": {
        "panel_A": "Bassez A metabolic tasks",
        "panel_B": "Bassez A GO biological processes",
        "panel_C": "Bassez B metabolic tasks",
        "panel_D": "Bassez B GO biological processes",
        "clustering": "independent row and column clustering preserved for every heatmap",
        "color_scales": "independent symmetric log2FC scales; direction comparable, magnitude not cross-modal",
        "bassez_group_labels": ["E", "NE"],
        "bassez_B_GO_limitation": "B-cell NE not tested because Pre n=15 is below the 20-cell threshold",
    },
    "go_heatmap": {
        "selection": "top 25 global-BH-significant terms by mean log2FC separately for E and NE",
        "matrix": "50 selection slots x available expansion-group/cell-type columns",
        "value": "raw On/Pre log2FC; no row scaling",
        "row_labels": "full GO names shown; trailing GO identifiers retained in data but omitted from the figure",
        "selection_direction": "strongest positive average On/Pre changes; individual cells may be negative",
        "audits": go_heatmap_component_audits,
        "matplotlib_fallback": True,
    },
    "rendering": {
        "S3_A": "fully vector",
        "S3_B": "fully vector",
        "S3_C": "UMAP point layers rasterized; text, axes, legends, and color bars vector",
    },
    "matrix_hashes": matrix_hashes,
    "interpretation": "external benchmarking; patient is biological replicate; pooled-cell GO statistics descriptive",
    "component_outputs": {
        "main": main_component_outputs,
        "supplementary": supplementary_component_outputs,
    },
    "compiled_outputs": {
        "main": main_outputs,
        **final_s3_outputs,
        "three_page_pdf": {"pdf": str(three_page_pdf)},
    },
    "mixed_media_audit": mixed_media_audit,
}
manifest_path.write_text(json.dumps(manifest_update, indent=2) + "\n")

print("Figure 3 version 2 integrated metabolic/GO heatmap validation passed")
display(pd.DataFrame({
    "compiled output": ["main", *final_s3_outputs],
    "formats": [", ".join(main_outputs), *[", ".join(v) for v in final_s3_outputs.values()]],
}))


## Benchmarking conclusion

The final panels benchmark external reproducibility and biological consistency. They do not convert descriptive pooled-cell statistics into independent patient-level discoveries.